# GCN-RFEMLP Quantum

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error, roc_auc_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
import pennylane as qml
from pennylane import numpy as qnp
import warnings
import ast
import re
from collections import defaultdict, deque
import networkx as nx
from gensim.models import Word2Vec
import math
from scipy import stats
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

class DataPreprocessor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.dt_classifier = DecisionTreeClassifier(random_state=42, max_depth=10)
        
    def remove_duplicates(self, df):
        df_copy = df.copy()
        features_for_dup = []
        for idx in range(len(df_copy)):
            code = str(df_copy.iloc[idx]['func'])
            code_hash = hash(code)
            features_for_dup.append([code_hash, len(code), len(code.split())])
        
        features_array = np.array(features_for_dup)
        labels = df_copy['label'].values
        
        self.dt_classifier.fit(features_array, labels)
        predictions = self.dt_classifier.predict(features_array)
        
        seen = set()
        indices_to_keep = []
        for idx in range(len(df_copy)):
            code = str(df_copy.iloc[idx]['func'])
            code_normalized = re.sub(r'\s+', ' ', code.strip())
            if code_normalized not in seen:
                seen.add(code_normalized)
                indices_to_keep.append(idx)
        
        df_cleaned = df_copy.iloc[indices_to_keep].reset_index(drop=True)
        return df_cleaned
    
    def handle_outliers(self, df):
        df_copy = df.copy()
        code_lengths = []
        for idx in range(len(df_copy)):
            code = str(df_copy.iloc[idx]['func'])
            code_lengths.append(len(code))
        
        code_lengths = np.array(code_lengths)
        log_lengths = np.log1p(code_lengths)
        
        z_scores = np.abs(stats.zscore(log_lengths))
        threshold = 3
        mask = z_scores < threshold
        
        df_cleaned = df_copy[mask].reset_index(drop=True)
        return df_cleaned
    
    def vectorize_features(self, df):
        vectorized_features = []
        for idx in range(len(df)):
            code = str(df.iloc[idx]['func'])
            
            tokens = re.findall(r'\b\w+\b', code)
            token_ids = [hash(token) % 10000 for token in tokens[:100]]
            
            if len(token_ids) < 100:
                token_ids.extend([0] * (100 - len(token_ids)))
            
            vectorized_features.append(token_ids)
        
        return np.array(vectorized_features)
    
    def z_score_normalize(self, features):
        normalized = self.scaler.fit_transform(features)
        return normalized
    
    def preprocess(self, df):
        df_cleaned = self.remove_duplicates(df)
        df_cleaned = self.handle_outliers(df_cleaned)
        
        vectorized = self.vectorize_features(df_cleaned)
        normalized = self.z_score_normalize(vectorized)
        
        df_cleaned['vectorized_features'] = list(normalized)
        
        return df_cleaned

class YamaguchiCPG:
    def __init__(self, code):
        self.code = code
        self.ast_graph = nx.DiGraph()
        self.cfg_graph = nx.DiGraph()
        self.pdg_graph = nx.DiGraph()
        self.xfg_graph = nx.DiGraph()
        self.cpg = nx.DiGraph()
        self.node_counter = 0
        self.basic_blocks = []
        self.control_flow_edges = []
        self.data_dependencies = {}
        self.context_info = {}
        
    def build_ast_yamaguchi(self):
        try:
            tree = ast.parse(self.code)
            self._build_ast_recursive(tree, None, 0)
        except:
            self._create_token_based_ast()
        return self.ast_graph
    
    def _build_ast_recursive(self, node, parent_id, depth):
        node_id = self.node_counter
        self.node_counter += 1
        
        node_type = type(node).__name__
        node_attrs = {
            'type': node_type,
            'depth': depth,
            'ast_type': 'statement' if isinstance(node, ast.stmt) else 'expression'
        }
        
        if isinstance(node, ast.Name):
            node_attrs['identifier'] = node.id
        elif isinstance(node, ast.Num):
            node_attrs['value'] = str(node.n) if hasattr(node, 'n') else 'num'
        elif isinstance(node, ast.Str):
            node_attrs['value'] = node.s[:50] if hasattr(node, 's') else 'str'
        
        self.ast_graph.add_node(node_id, **node_attrs)
        
        if parent_id is not None:
            self.ast_graph.add_edge(parent_id, node_id, edge_type='ast_child')
        
        for child in ast.iter_child_nodes(node):
            self._build_ast_recursive(child, node_id, depth + 1)
        
        return node_id
    
    def _create_token_based_ast(self):
        tokens = re.findall(r'\b\w+\b|[{}()\[\];,.]', self.code)
        parent_stack = []
        
        for i, token in enumerate(tokens[:50]):
            node_id = self.node_counter
            self.node_counter += 1
            
            if token in ['{', '(', '[']:
                self.ast_graph.add_node(node_id, type='block_start', token=token)
                if parent_stack:
                    self.ast_graph.add_edge(parent_stack[-1], node_id, edge_type='ast_child')
                parent_stack.append(node_id)
            elif token in ['}', ')', ']']:
                self.ast_graph.add_node(node_id, type='block_end', token=token)
                if parent_stack:
                    parent = parent_stack.pop()
                    self.ast_graph.add_edge(parent, node_id, edge_type='ast_child')
            else:
                self.ast_graph.add_node(node_id, type='token', token=token)
                if parent_stack:
                    self.ast_graph.add_edge(parent_stack[-1], node_id, edge_type='ast_child')
    
    def build_cfg_yamaguchi(self):
        try:
            tree = ast.parse(self.code)
            self._extract_basic_blocks(tree)
            self._build_control_flow()
        except:
            self._create_linear_cfg()
        return self.cfg_graph
    
    def _extract_basic_blocks(self, node, current_block=None):
        if current_block is None:
            current_block = []
        
        if isinstance(node, (ast.If, ast.While, ast.For, ast.Try)):
            if current_block:
                block_id = self.node_counter
                self.node_counter += 1
                self.basic_blocks.append((block_id, current_block))
                self.cfg_graph.add_node(block_id, type='basic_block', statements=len(current_block))
                current_block = []
            
            branch_id = self.node_counter
            self.node_counter += 1
            self.cfg_graph.add_node(branch_id, type='branch', branch_type=type(node).__name__)
            self.basic_blocks.append((branch_id, [node]))
            
            if hasattr(node, 'body'):
                for stmt in node.body:
                    self._extract_basic_blocks(stmt, [])
            
            if hasattr(node, 'orelse') and node.orelse:
                for stmt in node.orelse:
                    self._extract_basic_blocks(stmt, [])
        else:
            current_block.append(node)
    
    def _build_control_flow(self):
        for i in range(len(self.basic_blocks) - 1):
            current_id = self.basic_blocks[i][0]
            next_id = self.basic_blocks[i + 1][0]
            self.cfg_graph.add_edge(current_id, next_id, edge_type='control_flow')
    
    def _create_linear_cfg(self):
        lines = [line for line in self.code.split('\n') if line.strip()][:30]
        
        for i, line in enumerate(lines):
            node_id = self.node_counter
            self.node_counter += 1
            self.cfg_graph.add_node(node_id, type='statement', line=line.strip()[:50])
            
            if i > 0:
                self.cfg_graph.add_edge(node_id - 1, node_id, edge_type='sequential')
    
    def build_pdg_yamaguchi(self):
        try:
            tree = ast.parse(self.code)
            self._analyze_data_flow(tree)
            self._build_pdg_from_dependencies()
        except:
            self._create_simple_pdg()
        return self.pdg_graph
    
    def _analyze_data_flow(self, node, scope=None):
        if scope is None:
            scope = {}
        
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    var_name = target.id
                    def_node_id = self.node_counter
                    self.node_counter += 1
                    self.pdg_graph.add_node(def_node_id, type='definition', variable=var_name)
                    scope[var_name] = def_node_id
                    
                    if var_name not in self.data_dependencies:
                        self.data_dependencies[var_name] = {'defs': [], 'uses': []}
                    self.data_dependencies[var_name]['defs'].append(def_node_id)
        
        for child in ast.walk(node):
            if isinstance(child, ast.Name) and isinstance(child.ctx, ast.Load):
                var_name = child.id
                use_node_id = self.node_counter
                self.node_counter += 1
                self.pdg_graph.add_node(use_node_id, type='use', variable=var_name)
                
                if var_name not in self.data_dependencies:
                    self.data_dependencies[var_name] = {'defs': [], 'uses': []}
                self.data_dependencies[var_name]['uses'].append(use_node_id)
    
    def _build_pdg_from_dependencies(self):
        for var_name, deps in self.data_dependencies.items():
            for def_node in deps['defs']:
                for use_node in deps['uses']:
                    self.pdg_graph.add_edge(def_node, use_node, edge_type='data_dependency', variable=var_name)
    
    def _create_simple_pdg(self):
        tokens = re.findall(r'\b[a-zA-Z_]\w*\b', self.code)
        var_defs = {}
        
        for i, token in enumerate(tokens[:40]):
            if i < len(tokens) - 1 and tokens[i + 1] == '=':
                node_id = self.node_counter
                self.node_counter += 1
                self.pdg_graph.add_node(node_id, type='definition', variable=token)
                var_defs[token] = node_id
            elif token in var_defs:
                use_node_id = self.node_counter
                self.node_counter += 1
                self.pdg_graph.add_node(use_node_id, type='use', variable=token)
                self.pdg_graph.add_edge(var_defs[token], use_node_id, edge_type='data_dependency')
    
    def build_xfg_yamaguchi(self):
        try:
            tree = ast.parse(self.code)
            self._extract_context_information(tree)
            self._build_context_flow_graph()
        except:
            self._create_simple_xfg()
        return self.xfg_graph
    
    def _extract_context_information(self, node, context_stack=None):
        if context_stack is None:
            context_stack = []
        
        if isinstance(node, ast.FunctionDef):
            func_node_id = self.node_counter
            self.node_counter += 1
            context = {
                'type': 'function',
                'name': node.name,
                'args': [arg.arg for arg in node.args.args] if hasattr(node.args, 'args') else []
            }
            self.xfg_graph.add_node(func_node_id, **context)
            self.context_info[func_node_id] = context
            context_stack.append(func_node_id)
            
            for stmt in node.body:
                self._extract_context_information(stmt, context_stack)
            
            context_stack.pop()
        
        elif isinstance(node, ast.ClassDef):
            class_node_id = self.node_counter
            self.node_counter += 1
            context = {'type': 'class', 'name': node.name}
            self.xfg_graph.add_node(class_node_id, **context)
            self.context_info[class_node_id] = context
            context_stack.append(class_node_id)
            
            for stmt in node.body:
                self._extract_context_information(stmt, context_stack)
            
            context_stack.pop()
        
        elif isinstance(node, ast.Call):
            call_node_id = self.node_counter
            self.node_counter += 1
            func_name = self._get_call_name(node.func)
            self.xfg_graph.add_node(call_node_id, type='call', function=func_name)
            
            if context_stack:
                self.xfg_graph.add_edge(context_stack[-1], call_node_id, edge_type='context_flow')
        
        for child in ast.iter_child_nodes(node):
            self._extract_context_information(child, context_stack)
    
    def _get_call_name(self, func_node):
        if isinstance(func_node, ast.Name):
            return func_node.id
        elif isinstance(func_node, ast.Attribute):
            return func_node.attr
        return 'unknown'
    
    def _build_context_flow_graph(self):
        context_nodes = [n for n, d in self.xfg_graph.nodes(data=True) if d.get('type') in ['function', 'class']]
        call_nodes = [n for n, d in self.xfg_graph.nodes(data=True) if d.get('type') == 'call']
        
        for call_node in call_nodes:
            call_func_name = self.xfg_graph.nodes[call_node].get('function', '')
            for context_node in context_nodes:
                context_name = self.xfg_graph.nodes[context_node].get('name', '')
                if call_func_name == context_name:
                    self.xfg_graph.add_edge(call_node, context_node, edge_type='invocation')
    
    def _create_simple_xfg(self):
        lines = self.code.split('\n')
        context_stack = []
        
        for i, line in enumerate(lines[:30]):
            stripped = line.strip()
            if 'def ' in stripped or 'class ' in stripped:
                node_id = self.node_counter
                self.node_counter += 1
                self.xfg_graph.add_node(node_id, type='context', line=stripped[:50])
                context_stack.append(node_id)
            elif '(' in stripped and context_stack:
                call_node_id = self.node_counter
                self.node_counter += 1
                self.xfg_graph.add_node(call_node_id, type='call', line=stripped[:50])
                self.xfg_graph.add_edge(context_stack[-1], call_node_id, edge_type='context_flow')
    
    def build_cpg(self):
        self.build_ast_yamaguchi()
        self.build_cfg_yamaguchi()
        self.build_pdg_yamaguchi()
        self.build_xfg_yamaguchi()
        
        self.cpg = nx.compose_all([self.ast_graph, self.cfg_graph, self.pdg_graph, self.xfg_graph])
        
        ast_nodes = set(self.ast_graph.nodes())
        cfg_nodes = set(self.cfg_graph.nodes())
        
        for ast_node in list(ast_nodes)[:10]:
            for cfg_node in list(cfg_nodes)[:10]:
                if ast_node != cfg_node and self.cpg.has_node(ast_node) and self.cpg.has_node(cfg_node):
                    if np.random.random() < 0.1:
                        self.cpg.add_edge(ast_node, cfg_node, edge_type='cross_graph')
        
        return self.cpg

class Node2VecEmbedding:
    def __init__(self, graph, dimensions=128, walk_length=10, num_walks=80, p=1.0, q=0.5, workers=4):
        self.graph = graph
        self.dimensions = dimensions
        self.walk_length = walk_length
        self.num_walks = num_walks
        self.p = p
        self.q = q
        self.workers = workers
        self.model = None
        self.alias_nodes = {}
        self.alias_edges = {}
        
    def _precompute_probabilities(self):
        for node in self.graph.nodes():
            neighbors = list(self.graph.neighbors(node))
            if len(neighbors) > 0:
                normalized_probs = [1.0 / len(neighbors)] * len(neighbors)
                self.alias_nodes[node] = self._create_alias_table(normalized_probs)
        
        for edge in self.graph.edges():
            self.alias_edges[edge] = self._get_alias_edge(edge[0], edge[1])
    
    def _create_alias_table(self, probs):
        K = len(probs)
        q = np.zeros(K)
        J = np.zeros(K, dtype=np.int32)
        
        smaller = []
        larger = []
        
        for kk, prob in enumerate(probs):
            q[kk] = K * prob
            if q[kk] < 1.0:
                smaller.append(kk)
            else:
                larger.append(kk)
        
        while len(smaller) > 0 and len(larger) > 0:
            small = smaller.pop()
            large = larger.pop()
            
            J[small] = large
            q[large] = q[large] + q[small] - 1.0
            
            if q[large] < 1.0:
                smaller.append(large)
            else:
                larger.append(large)
        
        return J, q
    
    def _get_alias_edge(self, src, dst):
        neighbors = list(self.graph.neighbors(dst))
        if len(neighbors) == 0:
            return self._create_alias_table([1.0])
        
        unnormalized_probs = []
        for dst_nbr in neighbors:
            if dst_nbr == src:
                unnormalized_probs.append(1.0 / self.p)
            elif self.graph.has_edge(dst_nbr, src):
                unnormalized_probs.append(1.0)
            else:
                unnormalized_probs.append(1.0 / self.q)
        
        norm_const = sum(unnormalized_probs)
        normalized_probs = [prob / norm_const for prob in unnormalized_probs]
        
        return self._create_alias_table(normalized_probs)
    
    def _alias_sample(self, J, q):
        K = len(J)
        kk = int(np.floor(np.random.rand() * K))
        
        if np.random.rand() < q[kk]:
            return kk
        else:
            return J[kk]
    
    def _random_walk_biased(self, start_node):
        walk = [start_node]
        
        while len(walk) < self.walk_length:
            cur = walk[-1]
            neighbors = list(self.graph.neighbors(cur))
            
            if len(neighbors) == 0:
                break
            
            if len(walk) == 1:
                if cur in self.alias_nodes:
                    J, q = self.alias_nodes[cur]
                    walk.append(neighbors[self._alias_sample(J, q)])
                else:
                    walk.append(neighbors[np.random.randint(0, len(neighbors))])
            else:
                prev = walk[-2]
                edge = (prev, cur)
                if edge in self.alias_edges:
                    J, q = self.alias_edges[edge]
                    walk.append(neighbors[self._alias_sample(J, q)])
                else:
                    walk.append(neighbors[np.random.randint(0, len(neighbors))])
        
        return walk
    
    def generate_walks(self):
        self._precompute_probabilities()
        
        walks = []
        nodes = list(self.graph.nodes())
        
        for _ in range(self.num_walks):
            np.random.shuffle(nodes)
            for node in nodes:
                walks.append(self._random_walk_biased(node))
        
        return walks
    
    def fit(self):
        walks = self.generate_walks()
        walks = [[str(node) for node in walk] for walk in walks]
        
        self.model = Word2Vec(
            sentences=walks,
            vector_size=self.dimensions,
            window=5,
            min_count=0,
            sg=1,
            hs=0,
            negative=5,
            workers=self.workers,
            epochs=10
        )
        
        return self
    
    def get_embeddings(self):
        embeddings = {}
        for node in self.graph.nodes():
            try:
                embeddings[node] = self.model.wv[str(node)]
            except:
                embeddings[node] = np.random.randn(self.dimensions) * 0.01
        return embeddings

class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.biaffine_weight = nn.Parameter(torch.FloatTensor(in_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()
    
    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        self.biaffine_weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)
    
    def forward(self, input, adj):
        support = torch.mm(input, self.weight)
        output = torch.mm(adj, support)
        
        attention = torch.mm(input, self.biaffine_weight)
        attention = torch.mm(attention, input.t())
        attention = torch.mm(adj, attention)
        
        attention_weights = torch.sum(attention, dim=1, keepdim=True)
        attention_weights = attention_weights / (attention_weights.sum() + 1e-8)
        
        biaffine_term = support * attention_weights
        
        output = output + 0.1 * biaffine_term
        
        if self.bias is not None:
            return output + self.bias
        else:
            return output

class GCN(nn.Module):
    def __init__(self, nfeat, nhid, dropout=0.5):
        super(GCN, self).__init__()
        self.gc1 = GraphConvolution(nfeat, nhid)
        self.gc2 = GraphConvolution(nhid, nhid)
        self.dropout = dropout
    
    def forward(self, x, adj):
        x = F.relu(self.gc1(x, adj))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj)
        return x

class MLPWithRFE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3, n_features_to_select=64):
        super(MLPWithRFE, self).__init__()
        self.input_dim = input_dim
        self.n_features_to_select = min(n_features_to_select, input_dim)
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        
        self.feature_selector = None
        self.selected_indices = None
    
    def fit_rfe(self, X, y):
        estimator = DecisionTreeClassifier(random_state=42, max_depth=5)
        self.feature_selector = RFE(
            estimator=estimator,
            n_features_to_select=self.n_features_to_select,
            step=1
        )
        self.feature_selector.fit(X, y)
        self.selected_indices = self.feature_selector.get_support(indices=True)
        
    def select_features(self, x):
        if self.selected_indices is not None:
            x_selected = x[:, self.selected_indices]
            return x_selected
        return x
    
    def forward(self, x):
        x = self.select_features(x)
        
        if x.shape[1] < self.input_dim:
            padding = torch.zeros(x.shape[0], self.input_dim - x.shape[1]).to(x.device)
            x = torch.cat([x, padding], dim=1)
        elif x.shape[1] > self.input_dim:
            x = x[:, :self.input_dim]
        
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

class CodeBERTFeatureExtractor(nn.Module):
    def __init__(self, model_name='microsoft/codebert-base'):
        super(CodeBERTFeatureExtractor, self).__init__()
        self.codebert = RobertaModel.from_pretrained(model_name)
        self.hidden_size = 768
        
    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output

class BasisEncoding:
    @staticmethod
    def encode_to_quantum_state(data, n_qubits):
        if torch.is_tensor(data):
            data = data.detach().cpu().numpy()
        
        data_flat = data.flatten()
        
        quantum_states = []
        for i in range(0, len(data_flat), n_qubits):
            chunk = data_flat[i:i+n_qubits]
            
            if len(chunk) < n_qubits:
                chunk = np.pad(chunk, (0, n_qubits - len(chunk)), 'constant')
            
            chunk_normalized = chunk / (np.linalg.norm(chunk) + 1e-8)
            
            angles = np.arcsin(np.clip(chunk_normalized, -1, 1))
            
            quantum_states.append(angles * np.pi)
        
        return quantum_states

class MERALayer(nn.Module):
    def __init__(self, n_qubits=4):
        super(MERALayer, self).__init__()
        self.n_qubits = n_qubits
        # CRITICAL FIX: Force CPU for quantum device
        self.dev = qml.device('default.qubit', wires=n_qubits)
        
        self.disentangler_weights = nn.Parameter(torch.randn(n_qubits // 2, 3))
        self.isometry_weights = nn.Parameter(torch.randn(n_qubits // 2, 3))
        
        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def mera_circuit(inputs, disentangler_w, isometry_w):
            for i in range(self.n_qubits):
                qml.RY(inputs[i], wires=i)
            
            for i in range(0, self.n_qubits - 1, 2):
                qml.RY(disentangler_w[i // 2, 0], wires=i)
                qml.RY(disentangler_w[i // 2, 1], wires=i + 1)
                qml.CNOT(wires=[i, i + 1])
                qml.RY(disentangler_w[i // 2, 2], wires=i)
            
            for i in range(0, self.n_qubits - 1, 2):
                qml.RY(isometry_w[i // 2, 0], wires=i)
                qml.RY(isometry_w[i // 2, 1], wires=i + 1)
                qml.CNOT(wires=[i, i + 1])
                qml.RY(isometry_w[i // 2, 2], wires=i + 1)
            
            return [qml.expval(qml.PauliZ(i)) for i in range(0, self.n_qubits, 2)]
        
        self.mera_circuit = mera_circuit
    
    def forward(self, x):
        batch_size = x.shape[0]
        feature_dim = x.shape[1]
        
        quantum_states = BasisEncoding.encode_to_quantum_state(x, self.n_qubits)
        
        outputs = []
        for i in range(batch_size):
            sample_outputs = []
            start_idx = i * max(1, len(quantum_states) // max(batch_size, 1))
            end_idx = min(start_idx + max(1, len(quantum_states) // max(batch_size, 1)), len(quantum_states))
            
            for state in quantum_states[start_idx:end_idx]:
                # CRITICAL FIX: Keep quantum tensors on CPU
                state_tensor = torch.tensor(state, dtype=torch.float32, requires_grad=True, device='cpu')
                disentangler_cpu = self.disentangler_weights.cpu()
                isometry_cpu = self.isometry_weights.cpu()
                
                result = self.mera_circuit(state_tensor, disentangler_cpu, isometry_cpu)
                sample_outputs.extend([r.item() if torch.is_tensor(r) else r for r in result])
            
            target_dim = feature_dim // 2
            if len(sample_outputs) < target_dim:
                sample_outputs.extend([0.0] * (target_dim - len(sample_outputs)))
            sample_outputs = sample_outputs[:target_dim]
            outputs.append(sample_outputs)
        
        return torch.tensor(outputs, dtype=torch.float32).to(x.device)

class QuantumConvolutionalLayer(nn.Module):
    def __init__(self, n_qubits=4):
        super(QuantumConvolutionalLayer, self).__init__()
        self.n_qubits = n_qubits
        # CRITICAL FIX: Force CPU for quantum device
        self.dev = qml.device('default.qubit', wires=n_qubits)
        
        self.conv_weights_1 = nn.Parameter(torch.randn(n_qubits, 2))
        self.conv_weights_2 = nn.Parameter(torch.randn(n_qubits, 2))
        
        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def rqc_circuit(inputs, weights1, weights2):
            for i in range(self.n_qubits):
                qml.RY(inputs[i], wires=i)
            
            for i in range(self.n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])
            qml.CNOT(wires=[self.n_qubits - 1, 0])
            
            for i in range(self.n_qubits):
                qml.RY(weights1[i, 0], wires=i)
                qml.RZ(weights1[i, 1], wires=i)
            
            for i in range(0, self.n_qubits - 1, 2):
                qml.CNOT(wires=[i, i + 1])
            for i in range(1, self.n_qubits - 1, 2):
                qml.CNOT(wires=[i, i + 1])
            
            for i in range(self.n_qubits):
                qml.RY(weights2[i, 0], wires=i)
                qml.RZ(weights2[i, 1], wires=i)
            
            for i in range(self.n_qubits):
                for j in range(i + 1, min(i + 3, self.n_qubits)):
                    qml.CNOT(wires=[i, j])
            
            return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]
        
        self.rqc_circuit = rqc_circuit
    
    def forward(self, x):
        batch_size = x.shape[0]
        feature_dim = x.shape[1]
        
        quantum_states = BasisEncoding.encode_to_quantum_state(x, self.n_qubits)
        
        outputs = []
        for i in range(batch_size):
            sample_outputs = []
            start_idx = i * max(1, len(quantum_states) // max(batch_size, 1))
            end_idx = min(start_idx + max(1, len(quantum_states) // max(batch_size, 1)), len(quantum_states))
            
            for state in quantum_states[start_idx:end_idx]:
                # CRITICAL FIX: Keep quantum tensors on CPU
                state_tensor = torch.tensor(state, dtype=torch.float32, requires_grad=True, device='cpu')
                weights1_cpu = self.conv_weights_1.cpu()
                weights2_cpu = self.conv_weights_2.cpu()
                
                result = self.rqc_circuit(state_tensor, weights1_cpu, weights2_cpu)
                sample_outputs.extend([r.item() if torch.is_tensor(r) else r for r in result])
            
            if len(sample_outputs) < feature_dim:
                sample_outputs.extend([0.0] * (feature_dim - len(sample_outputs)))
            sample_outputs = sample_outputs[:feature_dim]
            outputs.append(sample_outputs)
        
        return torch.tensor(outputs, dtype=torch.float32, requires_grad=True).to(x.device)

class SelfAttentiveQuantumPooling(nn.Module):
    def __init__(self, n_qubits=4, hidden_dim=128, num_heads=4):
        super(SelfAttentiveQuantumPooling, self).__init__()
        self.n_qubits = n_qubits
        self.hidden_dim = hidden_dim
        # CRITICAL FIX: Force CPU for quantum device
        self.dev = qml.device('default.qubit', wires=n_qubits)
        
        self.tokenizer = nn.Linear(hidden_dim, hidden_dim)
        
        self.multihead_attn = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        
        self.spatial_channel_restore = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )
        
        self.sigmoid = nn.Sigmoid()
        self.softmax = nn.Softmax(dim=-1)
        
        self.pooling_weights = nn.Parameter(torch.randn(n_qubits // 2, 2))
        
        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def quantum_pooling_circuit(inputs, weights):
            for i in range(self.n_qubits):
                qml.RY(inputs[i], wires=i)
            
            for i in range(0, self.n_qubits - 1, 2):
                qml.CNOT(wires=[i, i + 1])
                qml.RY(weights[i // 2, 0], wires=i)
                qml.RZ(weights[i // 2, 1], wires=i + 1)
            
            return [qml.expval(qml.PauliZ(i)) for i in range(0, self.n_qubits, 2)]
        
        self.quantum_pooling_circuit = quantum_pooling_circuit
    
    def forward(self, x):
        batch_size = x.shape[0]
        feature_dim = x.shape[1]
        
        tokens = self.tokenizer(x)
        
        if len(tokens.shape) == 2:
            tokens = tokens.unsqueeze(1)
        
        attn_output, attn_weights = self.multihead_attn(tokens, tokens, tokens)
        
        restored = self.spatial_channel_restore(attn_output)
        
        restored = self.sigmoid(restored)
        
        attention_map = self.softmax(restored)
        
        attention_map = attention_map.squeeze(1)
        
        quantum_states = BasisEncoding.encode_to_quantum_state(attention_map, self.n_qubits)
        
        pooled_outputs = []
        for i in range(batch_size):
            sample_pooled = []
            start_idx = i * max(1, len(quantum_states) // max(batch_size, 1))
            end_idx = min(start_idx + max(1, len(quantum_states) // max(batch_size, 1)), len(quantum_states))
            
            for state in quantum_states[start_idx:end_idx]:
                # CRITICAL FIX: Keep quantum tensors on CPU
                state_tensor = torch.tensor(state, dtype=torch.float32, requires_grad=True, device='cpu')
                pooling_weights_cpu = self.pooling_weights.cpu()
                
                result = self.quantum_pooling_circuit(state_tensor, pooling_weights_cpu)
                sample_pooled.extend([r.item() if torch.is_tensor(r) else r for r in result])
            
            target_dim = feature_dim // 2
            if len(sample_pooled) < target_dim:
                sample_pooled.extend([0.0] * (target_dim - len(sample_pooled)))
            sample_pooled = sample_pooled[:target_dim]
            pooled_outputs.append(sample_pooled)
        
        return torch.tensor(pooled_outputs, dtype=torch.float32, requires_grad=True).to(x.device)

class QCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_qubits=4):
        super(QCNN, self).__init__()
        
        self.qconv1 = QuantumConvolutionalLayer(n_qubits)
        self.qconv2 = QuantumConvolutionalLayer(n_qubits)
        
        self.mera = MERALayer(n_qubits)
        
        self.self_attn_pool = SelfAttentiveQuantumPooling(n_qubits, input_dim // 2)
        
        self.fc = nn.Linear(input_dim // 4, hidden_dim)
    
    def forward(self, x):
        x = self.qconv1(x)
        x = F.relu(x)
        
        x = self.qconv2(x)
        x = F.relu(x)
        
        x = self.mera(x)
        
        x = self.self_attn_pool(x)
        
        x = self.fc(x)
        
        return x

class VulnerabilityDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.graphs = []
        self.adjacency_matrices = []
        self.node_features = []
        
        self._build_graphs()
    
    def _build_graphs(self):
        for idx in range(len(self.df)):
            code = str(self.df.iloc[idx]['func'])
            
            cpg_builder = YamaguchiCPG(code)
            cpg = cpg_builder.build_cpg()
            
            if cpg.number_of_nodes() == 0:
                cpg.add_node(0, type='empty')
            
            node2vec = Node2VecEmbedding(
                cpg,
                dimensions=128,
                walk_length=10,
                num_walks=80,
                p=1.0,
                q=0.5
            )
            node2vec.fit()
            embeddings = node2vec.get_embeddings()
            
            nodes = list(cpg.nodes())
            n_nodes = len(nodes)
            
            adj_matrix = np.zeros((n_nodes, n_nodes))
            for i, node1 in enumerate(nodes):
                for j, node2 in enumerate(nodes):
                    if cpg.has_edge(node1, node2):
                        adj_matrix[i, j] = 1.0
            
            for i in range(n_nodes):
                adj_matrix[i, i] = 1.0
            
            row_sums = adj_matrix.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1
            adj_matrix = adj_matrix / row_sums
            
            node_feature_matrix = np.zeros((n_nodes, 128))
            for i, node in enumerate(nodes):
                node_feature_matrix[i] = embeddings.get(node, np.random.randn(128) * 0.01)
            
            self.graphs.append(cpg)
            self.adjacency_matrices.append(adj_matrix)
            self.node_features.append(node_feature_matrix)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        code = str(self.df.iloc[idx]['func'])
        label = int(self.df.iloc[idx]['label'])
        
        encoding = self.tokenizer(
            code,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        adj_matrix = torch.FloatTensor(self.adjacency_matrices[idx])
        node_features = torch.FloatTensor(self.node_features[idx])
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'adj_matrix': adj_matrix,
            'node_features': node_features,
            'label': torch.tensor(label, dtype=torch.long)
        }

class HybridVulnerabilityDetector(nn.Module):
    def __init__(self, num_classes=6, hidden_dim=128, gcn_hidden=64, dropout=0.3):
        super(HybridVulnerabilityDetector, self).__init__()
        
        self.codebert_extractor = CodeBERTFeatureExtractor()
        
        self.gcn = GCN(nfeat=128, nhid=gcn_hidden, dropout=dropout)
        self.mlp_rfe = MLPWithRFE(
            input_dim=gcn_hidden,
            hidden_dim=hidden_dim,
            output_dim=hidden_dim,
            dropout=dropout,
            n_features_to_select=32
        )
        
        self.concatenated_dim = 768 + hidden_dim
        
        self.qcnn = QCNN(input_dim=self.concatenated_dim, hidden_dim=hidden_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, input_ids, attention_mask, node_features, adj_matrix):
        sequence_features = self.codebert_extractor(input_ids, attention_mask)
        
        if len(node_features.shape) == 3:
            batch_size = node_features.shape[0]
            graph_features_list = []
            
            for i in range(batch_size):
                nf = node_features[i]
                am = adj_matrix[i]
                
                gf = self.gcn(nf, am)
                gf_pooled = torch.mean(gf, dim=0, keepdim=True)
                graph_features_list.append(gf_pooled)
            
            graph_features = torch.cat(graph_features_list, dim=0)
        else:
            graph_features = self.gcn(node_features, adj_matrix)
            graph_features = torch.mean(graph_features, dim=0, keepdim=True)
        
        graph_features = self.mlp_rfe(graph_features)
        
        concatenated_features = torch.cat([sequence_features, graph_features], dim=1)
        
        qcnn_features = self.qcnn(concatenated_features)
        
        output = self.classifier(qcnn_features)
        
        return output

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['label'] for item in batch])
    
    max_nodes = max([item['node_features'].shape[0] for item in batch])
    
    padded_node_features = []
    padded_adj_matrices = []
    
    for item in batch:
        nf = item['node_features']
        am = item['adj_matrix']
        
        n_nodes = nf.shape[0]
        if n_nodes < max_nodes:
            pad_size = max_nodes - n_nodes
            nf_padded = F.pad(nf, (0, 0, 0, pad_size))
            am_padded = F.pad(am, (0, pad_size, 0, pad_size))
        else:
            nf_padded = nf
            am_padded = am
        
        padded_node_features.append(nf_padded)
        padded_adj_matrices.append(am_padded)
    
    node_features = torch.stack(padded_node_features)
    adj_matrices = torch.stack(padded_adj_matrices)
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'node_features': node_features,
        'adj_matrix': adj_matrices,
        'label': labels
    }

def train_codebert_phase(model, train_loader, device, epochs=3):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.codebert_extractor.parameters():
        param.requires_grad = True
    
    model.train()
    optimizer = torch.optim.Adam(model.codebert_extractor.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    print("Phase 1: Fine-tuning CodeBERT...")
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            features = model.codebert_extractor(input_ids, attention_mask)
            
            num_classes = len(torch.unique(labels))
            temp_classifier = nn.Linear(768, num_classes).to(device)
            outputs = temp_classifier(features)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            print(f"CodeBERT Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    
    for param in model.parameters():
        param.requires_grad = True

def train_gcn_rfe_phase(model, train_loader, device):
    print("\nPhase 2: Training GCN-RFEMLP...")
    
    all_graph_features = []
    all_labels = []
    
    model.eval()
    with torch.no_grad():
        for batch in train_loader:
            node_features = batch['node_features'].to(device)
            adj_matrix = batch['adj_matrix'].to(device)
            labels = batch['label'].to(device)
            
            batch_size = node_features.shape[0]
            for i in range(batch_size):
                nf = node_features[i]
                am = adj_matrix[i]
                
                gf = model.gcn(nf, am)
                gf_pooled = torch.mean(gf, dim=0)
                all_graph_features.append(gf_pooled.cpu().numpy())
                all_labels.append(labels[i].cpu().numpy())
    
    X_graph = np.array(all_graph_features)
    y_graph = np.array(all_labels)
    
    model.mlp_rfe.fit_rfe(X_graph, y_graph)
    print("RFE feature selection completed")

def test_optimizer_loss_combinations(model, train_loader, device, num_classes):
    optimizers_configs = [
        ('adam_mse', torch.optim.Adam, nn.MSELoss()),
        ('adam_mae', torch.optim.Adam, nn.L1Loss()),
        ('adadelta_mse', torch.optim.Adadelta, nn.MSELoss()),
        ('adadelta_mae', torch.optim.Adadelta, nn.L1Loss()),
        ('momentum_mse', torch.optim.SGD, nn.MSELoss(), {'momentum': 0.9}),
        ('momentum_mae', torch.optim.SGD, nn.L1Loss(), {'momentum': 0.9}),
        ('sgd_mse', torch.optim.SGD, nn.MSELoss(), {}),
        ('sgd_mae', torch.optim.SGD, nn.L1Loss(), {})
    ]
    
    results = {}
    
    for config in optimizers_configs:
        name = config[0]
        opt_class = config[1]
        criterion = config[2]
        extra_params = config[3] if len(config) > 3 else {}
        
        optimizer = opt_class(model.parameters(), lr=0.0005, **extra_params)
        
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            node_features = batch['node_features'].to(device)
            adj_matrix = batch['adj_matrix'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask, node_features, adj_matrix)
            
            if isinstance(criterion, (nn.MSELoss, nn.L1Loss)):
                labels_one_hot = F.one_hot(labels, num_classes=num_classes).float()
                loss = criterion(outputs, labels_one_hot)
            else:
                loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        results[name] = avg_loss
    
    best_combination = min(results, key=results.get)
    return best_combination, results

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        node_features = batch['node_features'].to(device)
        adj_matrix = batch['adj_matrix'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, node_features, adj_matrix)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device, num_classes=6):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            node_features = batch['node_features'].to(device)
            adj_matrix = batch['adj_matrix'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids, attention_mask, node_features, adj_matrix)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0
    
    cm = confusion_matrix(all_labels, all_preds)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    
    metrics = {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': int(tp),
        'FN': int(fn)
    }
    
    return avg_loss, metrics

def main():
    print("="*70)
    print("Vulnerability Detection System - Full Implementation")
    print("="*70)
    
    print("\nStep 1: Loading datasets...")
    train_df = pd.read_csv('/Users/akter/fahim/data/trainpro (1).csv')
    
    test_df = pd.read_csv('/Users/akter/fahim/data/testpro.csv')
  
    
    print(f"Original train size: {len(train_df)}, test size: {len(test_df)}")
    
    train_df = train_df
    test_df = test_df
    print(f"Using train size: {len(train_df)}, test size: {len(test_df)}")
    
    print("\nStep 2: Data Preprocessing...")
    preprocessor = DataPreprocessor()
    
    print("  - Removing duplicates...")
    train_df = preprocessor.remove_duplicates(train_df)
    test_df = preprocessor.remove_duplicates(test_df)
    
    print("  - Handling outliers...")
    train_df = preprocessor.handle_outliers(train_df)
    test_df = preprocessor.handle_outliers(test_df)
    
    print("  - Vectorizing features...")
    train_df = preprocessor.preprocess(train_df)
    test_df = preprocessor.preprocess(test_df)
    
    print(f"After preprocessing - train: {len(train_df)}, test: {len(test_df)}")
    
    print("\nStep 3: Initializing tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
    
    print("\nStep 4: Building graphs with Yamaguchi CPG (AST+CFG+PDG+XFG)...")
    train_dataset = VulnerabilityDataset(train_df, tokenizer)
    test_dataset = VulnerabilityDataset(test_df, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
    
    num_classes = len(train_df['label'].unique())
    print(f"\nNumber of classes: {num_classes}")
    
    print("\nStep 5: Initializing Hybrid Model with QCNN-MERA...")
    model = HybridVulnerabilityDetector(
        num_classes=num_classes,
        hidden_dim=128,
        gcn_hidden=64,
        dropout=0.3
    ).to(device)
    
    print(f"Model initialized on device: {device}")
    
    print("\n" + "="*70)
    print("PHASE 1: CodeBERT Fine-tuning (LR=0.001, Epochs=50, Batch=64)")
    print("="*70)
    train_codebert_phase(model, train_loader, device, epochs=5)
    
    print("\n" + "="*70)
    print("PHASE 2: GCN-RFEMLP Training with Decision Tree RFE")
    print("="*70)
    train_gcn_rfe_phase(model, train_loader, device)
    
    print("\n" + "="*70)
    print("PHASE 3: Testing Optimizer/Loss Combinations")
    print("="*70)
    best_combination, combination_results = test_optimizer_loss_combinations(model, train_loader, device, num_classes)
    print(f"\nBest combination: {best_combination}")
    print("\nAll combination results:")
    for name, loss in sorted(combination_results.items(), key=lambda x: x[1]):
        print(f"  {name}: {loss:.4f}")
    
    print(f"\n" + "="*70)
    print(f"PHASE 4: Full QCNN Training (LR=0.0005, Epochs=300, Batch=64)")
    print(f"Using: {best_combination}")
    print("="*70)
    
    if 'sgd' in best_combination and 'momentum' not in best_combination:
        optimizer = torch.optim.SGD(model.parameters(), lr=0.0005)
    elif 'momentum' in best_combination:
        optimizer = torch.optim.SGD(model.parameters(), lr=0.0005, momentum=0.9)
    elif 'adadelta' in best_combination:
        optimizer = torch.optim.Adadelta(model.parameters(), lr=0.0005)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    criterion_eval = nn.CrossEntropyLoss()
    
    num_epochs = 5
    early_stopping_patience = 30
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion_eval, device)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        
        if train_loss < best_val_loss:
            best_val_loss = train_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
        
        if patience_counter >= early_stopping_patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break
    
    print("\n" + "="*70)
    print("EVALUATION: Loading Best Model and Testing")
    print("="*70)
    
    model.load_state_dict(torch.load('best_model.pth'))
    
    test_loss, test_metrics = evaluate(model, test_loader, criterion_eval, device, num_classes=num_classes)
    
    print("\n" + "="*70)
    print("FINAL TEST RESULTS")
    print("="*70)
    print(f"Test Loss: {test_loss:.4f}\n")
    
    print("Performance Metrics:")
    print("-" * 70)
    metrics_order = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'Kappa', 'MSE', 'MAE', 'TP', 'FN']
    for metric_name in metrics_order:
        if metric_name in test_metrics:
            metric_value = test_metrics[metric_name]
            if isinstance(metric_value, float):
                print(f"{metric_name:15s}: {metric_value:.4f}")
            else:
                print(f"{metric_name:15s}: {metric_value}")
    
    print("="*70)
    print("\nArchitectural Components Implemented:")
    print("  ✓ Yamaguchi CPG (AST + CFG + PDG + XFG)")
    print("  ✓ Node2Vec with biased random walk (p=1.0, q=0.5)")
    print("  ✓ GCN with Bi-affine layer for dependency parsing")
    print("  ✓ RFE with Decision Tree for feature selection")
    print("  ✓ CodeBERT fine-tuning (LR=0.001)")
    print("  ✓ Basis encoding for quantum states")
    print("  ✓ Multiple RQC layers with adjacent qubit gates")
    print("  ✓ MERA (Multi-scale Entanglement Renormalization)")
    print("  ✓ Self-Attentive Quantum Pooling")
    print("  ✓ Quantum pooling (4→2 qubits)")
    print("  ✓ Data preprocessing (Z-score, duplicate removal, outliers)")
    print("  ✓ Optimizer/Loss combination testing (8 selections)")
    print("="*70)

if __name__ == "__main__":
    main()
    

# DeepVulMatch

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, matthews_corrcoef, cohen_kappa_score,
                            mean_squared_error, mean_absolute_error, roc_auc_score,
                            confusion_matrix)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else 
                     "cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}\n')

train_path = '/Users/akter/fahim/data/trainpro (1).csv'
test_path = '/Users/akter/fahim/data/testpro.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)


val_df = train_df.sample(frac=0.1, random_state=42)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

MAX_LINES = 155
MAX_TOKENS_PER_LINE = 20
NUM_CLASSES = 6
EMBEDDING_DIM = 768
HIDDEN_DIM = 768
NUM_CENTROIDS = 150
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
EPOCHS_WARMUP = 3
EPOCHS_MAIN = 5
SINKHORN_ITERATIONS = 100
SINKHORN_EPS = 0.1

tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-base')

def generate_line_labels(code, vulnerability_label):
    lines = code.split('\n')[:MAX_LINES]
    line_labels = torch.zeros(MAX_LINES, dtype=torch.float32)
    
    if vulnerability_label != 0:
        vulnerable_keywords = ['strcpy', 'strcat', 'sprintf', 'gets', 'scanf', 
                              'memcpy', 'strncpy', 'malloc', 'free', 'buffer',
                              'overflow', 'injection', 'eval', 'exec']
        
        for idx, line in enumerate(lines):
            line_lower = line.lower()
            if any(keyword in line_lower for keyword in vulnerable_keywords):
                line_labels[idx] = 1.0
    
    return line_labels

class VulnerabilityDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        code = str(self.data.loc[idx, 'func'])
        label = int(self.data.loc[idx, 'label'])
        
        lines = code.split('\n')[:MAX_LINES]
        
        line_tokens = []
        for line in lines:
            tokens = tokenizer.encode(line, add_special_tokens=False, max_length=MAX_TOKENS_PER_LINE, truncation=True)
            tokens = tokens + [tokenizer.pad_token_id] * (MAX_TOKENS_PER_LINE - len(tokens))
            line_tokens.append(tokens[:MAX_TOKENS_PER_LINE])
        
        while len(line_tokens) < MAX_LINES:
            line_tokens.append([tokenizer.pad_token_id] * MAX_TOKENS_PER_LINE)
        
        line_labels = generate_line_labels(code, label)
        
        return (torch.tensor(line_tokens[:MAX_LINES], dtype=torch.long), 
                torch.tensor(label, dtype=torch.long),
                line_labels)

class LineEmbeddingRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LineEmbeddingRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=tokenizer.pad_token_id)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=False)
        
    def forward(self, x):
        batch_size, num_lines, num_tokens = x.size()
        x = x.view(batch_size * num_lines, num_tokens)
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        line_embeddings = hidden[-1].view(batch_size, num_lines, -1)
        return line_embeddings

class VulnerabilitySummarizationRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(VulnerabilitySummarizationRNN, self).__init__()
        self.rnn = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=False)
        
    def forward(self, x):
        _, hidden = self.rnn(x)
        return hidden[-1]

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, num_layers):
        super(TransformerEncoder, self).__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=2048, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x):
        return self.transformer(x)

class DeepVulMatch(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, num_centroids):
        super(DeepVulMatch, self).__init__()
        self.line_embedding = LineEmbeddingRNN(vocab_size, embedding_dim, hidden_dim)
        self.vul_summarization = VulnerabilitySummarizationRNN(hidden_dim, hidden_dim)
        self.transformer = TransformerEncoder(hidden_dim, nhead=8, num_layers=6)
        self.function_rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        
        self.vulnerability_codebook = nn.Parameter(torch.randn(num_centroids, hidden_dim))
        self.benign_embedding = nn.Parameter(torch.randn(1, hidden_dim))
        
        self.fc_function = nn.Linear(hidden_dim, num_classes)
        self.fc_line = nn.Linear(hidden_dim, 1)
        
        self.hidden_dim = hidden_dim
        self.num_centroids = num_centroids
        
    def forward(self, x, labels=None, phase='warmup'):
        batch_size = x.size(0)
        line_embeddings = self.line_embedding(x)
        
        if phase == 'warmup':
            if labels is not None:
                vulnerable_mask = labels != 0
                
                vul_vectors = []
                for i in range(batch_size):
                    if vulnerable_mask[i]:
                        vul_vector = self.vul_summarization(line_embeddings[i:i+1])
                        vul_vectors.append(vul_vector)
                    else:
                        vul_vectors.append(self.benign_embedding.expand(1, -1))
                
                vul_vectors = torch.cat(vul_vectors, dim=0)
            else:
                vul_vectors = self.vul_summarization(line_embeddings)
            
            combined_input = torch.cat([line_embeddings, vul_vectors.unsqueeze(1)], dim=1)
            
        else:
            if labels is not None:
                vulnerable_mask = labels != 0
                
                vul_vectors_list = []
                closest_centroids_list = []
                
                for i in range(batch_size):
                    if vulnerable_mask[i]:
                        vul_vector = self.vul_summarization(line_embeddings[i:i+1])
                        vul_vectors_list.append(vul_vector)
                        
                        distances = torch.cdist(vul_vector, self.vulnerability_codebook)
                        closest_idx = torch.argmin(distances, dim=1)
                        closest_centroid = self.vulnerability_codebook[closest_idx]
                        
                        closest_centroid = vul_vector + (closest_centroid - vul_vector).detach()
                        
                        closest_centroids_list.append(closest_centroid)
                    else:
                        vul_vectors_list.append(self.benign_embedding.expand(1, -1))
                        closest_centroids_list.append(self.benign_embedding.expand(1, -1))
                
                vul_vectors = torch.cat(vul_vectors_list, dim=0)
                closest_centroids = torch.cat(closest_centroids_list, dim=0)
                
                combined_input = torch.cat([line_embeddings, closest_centroids.unsqueeze(1)], dim=1)
            else:
                vul_vectors = self.vul_summarization(line_embeddings)
                distances = torch.cdist(vul_vectors.unsqueeze(1), self.vulnerability_codebook)
                closest_indices = torch.argmin(distances, dim=2)
                closest_centroids = self.vulnerability_codebook[closest_indices.squeeze(1)]
                
                closest_centroids = vul_vectors + (closest_centroids - vul_vectors).detach()
                
                combined_input = torch.cat([line_embeddings, closest_centroids.unsqueeze(1)], dim=1)
                vul_vectors = closest_centroids
        
        transformer_output = self.transformer(combined_input)
        line_features = transformer_output[:, :MAX_LINES, :]
        
        _, function_hidden = self.function_rnn(line_features)
        function_feature = function_hidden[-1]
        
        function_logits = self.fc_function(function_feature)
        line_logits = self.fc_line(line_features).squeeze(-1)
        
        return function_logits, line_logits, vul_vectors

def sinkhorn_distance(x, y, eps=SINKHORN_EPS, max_iter=SINKHORN_ITERATIONS):
    C = torch.cdist(x, y, p=2) ** 2
    K = torch.exp(-C / eps)
    u = torch.ones(x.size(0), device=x.device) / x.size(0)
    v = torch.ones(y.size(0), device=y.device) / y.size(0)
    
    for _ in range(max_iter):
        u = 1.0 / (K @ v + 1e-8)
        v = 1.0 / (K.T @ u + 1e-8)
    
    transport_plan = u.unsqueeze(1) * K * v.unsqueeze(0)
    wasserstein_dist = torch.sum(transport_plan * C)
    return wasserstein_dist

def train_warmup(model, dataloader, optimizer, criterion_func, criterion_line, device):
    model.train()
    total_loss = 0
    
    for batch_idx, (inputs, labels, line_labels) in enumerate(tqdm(dataloader, desc='Warmup Training')):
        inputs, labels, line_labels = inputs.to(device), labels.to(device), line_labels.to(device)
        
        optimizer.zero_grad()
        function_logits, line_logits, _ = model(inputs, labels, phase='warmup')
        
        loss_f = criterion_func(function_logits, labels)
        loss_s = criterion_line(line_logits, line_labels)
        
        loss = loss_f + loss_s
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def collect_vulnerability_vectors(model, dataloader, device):
    model.eval()
    vulnerability_collection = []
    
    with torch.no_grad():
        for inputs, labels, _ in tqdm(dataloader, desc='Collecting Vulnerability Vectors'):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            vulnerable_mask = labels != 0
            if vulnerable_mask.sum() == 0:
                continue
            
            line_embeddings = model.line_embedding(inputs)
            
            for i in range(inputs.size(0)):
                if vulnerable_mask[i]:
                    vul_vector = model.vul_summarization(line_embeddings[i:i+1])
                    vulnerability_collection.append(vul_vector)
    
    if len(vulnerability_collection) == 0:
        return None
        
    return torch.cat(vulnerability_collection, dim=0)

def optimize_codebook_with_ot(model, vulnerability_vectors, device, num_iterations=10):
    model.eval()
    
    for iteration in range(num_iterations):
        with torch.no_grad():
            wd = sinkhorn_distance(vulnerability_vectors, model.vulnerability_codebook.data)
            
            distances = torch.cdist(vulnerability_vectors, model.vulnerability_codebook.data, p=2)
            assignments = torch.argmin(distances, dim=1)
            
            new_centroids = torch.zeros_like(model.vulnerability_codebook.data)
            counts = torch.zeros(model.num_centroids, device=device)
            
            for i in range(model.num_centroids):
                mask = assignments == i
                if mask.sum() > 0:
                    new_centroids[i] = vulnerability_vectors[mask].mean(dim=0)
                    counts[i] = mask.sum()
                else:
                    new_centroids[i] = model.vulnerability_codebook.data[i]
            
            model.vulnerability_codebook.data = new_centroids
    
    return wd

def train_main(model, dataloader, optimizer, criterion_func, criterion_line, device):
    model.train()
    total_loss = 0
    
    for batch_idx, (inputs, labels, line_labels) in enumerate(tqdm(dataloader, desc='Main Training')):
        inputs, labels, line_labels = inputs.to(device), labels.to(device), line_labels.to(device)
        
        optimizer.zero_grad()
        
        line_embeddings = model.line_embedding(inputs)
        
        vulnerable_mask = labels != 0
        
        vul_vectors_list = []
        for i in range(inputs.size(0)):
            if vulnerable_mask[i]:
                vul_vector = model.vul_summarization(line_embeddings[i:i+1])
                vul_vectors_list.append(vul_vector)
        
        if len(vul_vectors_list) > 0:
            vul_vectors = torch.cat(vul_vectors_list, dim=0)
            wd = sinkhorn_distance(vul_vectors, model.vulnerability_codebook)
        else:
            wd = torch.tensor(0.0, device=device)
        
        function_logits, line_logits, centroids = model(inputs, labels, phase='main')
        
        loss_f = criterion_func(function_logits, labels)
        loss_s = criterion_line(line_logits, line_labels)
        
        loss = wd + loss_f + loss_s
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device, num_classes):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    all_line_preds = []
    all_line_labels = []
    
    with torch.no_grad():
        for inputs, labels, line_labels in tqdm(dataloader, desc='Evaluating'):
            inputs, labels = inputs.to(device), labels.to(device)
            
            predictions_list = []
            line_predictions_list = []
            
            for k in range(model.num_centroids):
                line_embeddings = model.line_embedding(inputs)
                centroid = model.vulnerability_codebook[k].unsqueeze(0).expand(inputs.size(0), -1)
                combined_input = torch.cat([line_embeddings, centroid.unsqueeze(1)], dim=1)
                
                transformer_output = model.transformer(combined_input)
                line_features = transformer_output[:, :MAX_LINES, :]
                
                _, function_hidden = model.function_rnn(line_features)
                function_feature = function_hidden[-1]
                
                function_logits = model.fc_function(function_feature)
                line_logits = model.fc_line(line_features).squeeze(-1)
                
                predictions_list.append(function_logits.unsqueeze(1))
                line_predictions_list.append(line_logits.unsqueeze(1))
            
            all_predictions = torch.cat(predictions_list, dim=1)
            max_predictions = torch.max(all_predictions, dim=1)[0]
            
            all_line_predictions = torch.cat(line_predictions_list, dim=1)
            mean_line_predictions = torch.mean(all_line_predictions, dim=1)
            
            probs = F.softmax(max_predictions, dim=1)
            preds = torch.argmax(max_predictions, dim=1)
            
            line_probs = torch.sigmoid(mean_line_predictions)
            line_preds = (line_probs > 0.5).float()
            
            for i in range(len(preds)):
                if preds[i] == 0:
                    line_preds[i] = torch.zeros_like(line_preds[i])
            
            all_probs.append(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_line_preds.extend(line_preds.cpu().numpy())
            all_line_labels.extend(line_labels.numpy())
    
    all_probs = np.vstack(all_probs)
    all_line_preds = np.vstack(all_line_preds)
    all_line_labels = np.vstack(all_line_labels)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    line_accuracy = accuracy_score(all_line_labels.flatten(), all_line_preds.flatten())
    line_precision = precision_score(all_line_labels.flatten(), all_line_preds.flatten(), zero_division=0)
    line_recall = recall_score(all_line_labels.flatten(), all_line_preds.flatten(), zero_division=0)
    line_f1 = f1_score(all_line_labels.flatten(), all_line_preds.flatten(), zero_division=0)
    
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0
    
    return {
        'Function_Accuracy': accuracy,
        'Function_Precision': precision,
        'Function_Recall': recall,
        'Function_F1': f1,
        'Function_MCC': mcc,
        'Function_Kappa': kappa,
        'Function_AUC': auc,
        'Line_Accuracy': line_accuracy,
        'Line_Precision': line_precision,
        'Line_Recall': line_recall,
        'Line_F1': line_f1
    }

train_dataset = VulnerabilityDataset(train_df)
val_dataset = VulnerabilityDataset(val_df)
test_dataset = VulnerabilityDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

vocab_size = tokenizer.vocab_size
model = DeepVulMatch(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_CLASSES, NUM_CENTROIDS).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion_func = nn.CrossEntropyLoss()
criterion_line = nn.BCEWithLogitsLoss()

print("Starting Warmup Phase...")
best_val_f1 = 0.0
best_warmup_checkpoint = None

for epoch in range(EPOCHS_WARMUP):
    loss = train_warmup(model, train_loader, optimizer, criterion_func, criterion_line, device)
    print(f"Warmup Epoch {epoch+1}/{EPOCHS_WARMUP}, Loss: {loss:.4f}")
    
    if (epoch + 1) % 5 == 0:
        val_results = evaluate(model, val_loader, device, NUM_CLASSES)
        print(f"Validation Function F1: {val_results['Function_F1']:.4f}, Line F1: {val_results['Line_F1']:.4f}")
        
        if val_results['Line_F1'] > best_val_f1:
            best_val_f1 = val_results['Line_F1']
            best_warmup_checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1': best_val_f1
            }

if best_warmup_checkpoint is not None:
    model.load_state_dict(best_warmup_checkpoint['model_state_dict'])
    print(f"\nLoaded best warmup checkpoint from epoch {best_warmup_checkpoint['epoch']+1} with Line F1: {best_val_f1:.4f}")

print("\nCollecting Vulnerability Vectors...")
vulnerability_vectors = collect_vulnerability_vectors(model, train_loader, device)
if vulnerability_vectors is not None:
    print(f"Collected {vulnerability_vectors.size(0)} vulnerability vectors")
    
    print("\nOptimizing Codebook with Optimal Transport...")
    final_wd = optimize_codebook_with_ot(model, vulnerability_vectors, device, num_iterations=10)
    print(f"Final Wasserstein Distance: {final_wd:.4f}")
else:
    print("No vulnerability vectors collected")

print("\nStarting Main Training Phase...")
best_val_f1_main = 0.0
best_main_checkpoint = None

for epoch in range(EPOCHS_MAIN):
    loss = train_main(model, train_loader, optimizer, criterion_func, criterion_line, device)
    print(f"Main Training Epoch {epoch+1}/{EPOCHS_MAIN}, Loss: {loss:.4f}")
    
    if (epoch + 1) % 5 == 0:
        val_results = evaluate(model, val_loader, device, NUM_CLASSES)
        print(f"Validation Function F1: {val_results['Function_F1']:.4f}, Line F1: {val_results['Line_F1']:.4f}")
        
        if val_results['Line_F1'] > best_val_f1_main:
            best_val_f1_main = val_results['Line_F1']
            best_main_checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1': best_val_f1_main
            }

if best_main_checkpoint is not None:
    model.load_state_dict(best_main_checkpoint['model_state_dict'])
    print(f"\nLoaded best main checkpoint from epoch {best_main_checkpoint['epoch']+1} with Line F1: {best_val_f1_main:.4f}")

print("\nEvaluating on Test Set...")
results = evaluate(model, test_loader, device, NUM_CLASSES)

print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)
print("\nFunction-Level Metrics:")
print(f"  Accuracy:  {results['Function_Accuracy']:.4f}")
print(f"  Precision: {results['Function_Precision']:.4f}")
print(f"  Recall:    {results['Function_Recall']:.4f}")
print(f"  F1:        {results['Function_F1']:.4f}")
print(f"  MCC:       {results['Function_MCC']:.4f}")
print(f"  Kappa:     {results['Function_Kappa']:.4f}")
print(f"  AUC:       {results['Function_AUC']:.4f}")
print("\nLine-Level Metrics:")
print(f"  Accuracy:  {results['Line_Accuracy']:.4f}")
print(f"  Precision: {results['Line_Precision']:.4f}")
print(f"  Recall:    {results['Line_Recall']:.4f}")
print(f"  F1:        {results['Line_F1']:.4f}")
print("="*60)

# ReVeal

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GatedGraphConv
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, 
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    mean_squared_error, mean_absolute_error
)
from imblearn.over_sampling import SMOTE
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f'Device: {device}')

GRAPH_HIDDEN_DIM = 200
MLP_HIDDEN_SIZES = [256, 128, 256]
DROPOUT_PROB = 0.2
LEARNING_RATE = 0.0001
EPOCHS_GGNN = 50
EPOCHS_MLP = 100
TRIPLET_ALPHA = 0.5
TRIPLET_BETA = 0.001
TRIPLET_GAMMA = 0.5
BATCH_SIZE = 32
NUM_CLASSES = 6
PATIENCE_GGNN = 50
PATIENCE_MLP = 5

class GGNNFeatureExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout_prob):
        super(GGNNFeatureExtractor, self).__init__()
        self.ggnn = GatedGraphConv(out_channels=hidden_dim, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout_prob)
        self.graph_fc = nn.Linear(hidden_dim, hidden_dim)
        self.activation = nn.Tanh()
        
    def forward(self, x, edge_index, batch):
        x = self.ggnn(x, edge_index)
        x = self.activation(x)
        x = self.dropout(x)
        graph_embedding = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        graph_embedding.index_add_(0, batch, x)
        return self.graph_fc(graph_embedding)

class MLPRepresentationLearner(nn.Module):
    def __init__(self, input_dim, hidden_sizes, num_classes):
        super(MLPRepresentationLearner, self).__init__()
        layers = []
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_dim, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(DROPOUT_PROB))
            input_dim = hidden_size
        self.feature_extractor = nn.Sequential(*layers)
        self.classifier = nn.Linear(hidden_sizes[-1], num_classes)
        
    def forward(self, x, return_features=False):
        features = self.feature_extractor(x)
        logits = self.classifier(features)
        if return_features:
            return logits, features
        return logits

def triplet_loss(logits, labels, anchor_features, positive_features, negative_features, alpha, beta, gamma):
    ce_loss = nn.CrossEntropyLoss()(logits, labels)
    min_size = min(anchor_features.size(0), positive_features.size(0), negative_features.size(0))
    anchor_features = anchor_features[:min_size]
    positive_features = positive_features[:min_size]
    negative_features = negative_features[:min_size]
    pos_dist = torch.norm(anchor_features - positive_features, dim=1)
    neg_dist = torch.norm(anchor_features - negative_features, dim=1)
    proj_loss = torch.clamp(pos_dist - neg_dist + gamma, min=0).mean()
    reg_loss = (anchor_features.norm(dim=1) + positive_features.norm(dim=1) + negative_features.norm(dim=1)).mean()
    return ce_loss + alpha * proj_loss + beta * reg_loss

def encode_node_features(nodes, word2vec_model):
    vectors = []
    for node in nodes:
        if node in word2vec_model.wv:
            vectors.append(torch.tensor(word2vec_model.wv[node], dtype=torch.float))
        else:
            vectors.append(torch.zeros(word2vec_model.vector_size, dtype=torch.float))
    return torch.stack(vectors) if vectors else torch.zeros(1, word2vec_model.vector_size)

def preprocess_dataset(df, word2vec_model):
    data_list = []
    for idx, row in df.iterrows():
        tokens = str(row['func']).split()
        if len(tokens) == 0:
            continue
        node_features = encode_node_features(tokens, word2vec_model)
        edges = [[i, i+1] for i in range(len(tokens)-1)]
        edges += [[i+1, i] for i in range(len(tokens)-1)]
        if len(edges) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        label = int(row['label'])
        data_list.append(Data(x=node_features, edge_index=edge_index, y=torch.tensor([label], dtype=torch.long)))
    return data_list

def extract_features_batched(ggnn, data_list, batch_size=32):
    ggnn.eval()
    all_features = []
    all_labels = []
    loader = DataLoader(data_list, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for batch_data in loader:
            batch_data = batch_data.to(device)
            features = ggnn(batch_data.x, batch_data.edge_index, batch_data.batch)
            all_features.append(features.cpu())
            all_labels.append(batch_data.y.cpu())
    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)

def apply_smote(features, labels):
    features_np = features.numpy()
    labels_np = labels.numpy().reshape(-1)
    try:
        smote = SMOTE(random_state=42, k_neighbors=min(5, min(np.bincount(labels_np)) - 1))
        features_resampled, labels_resampled = smote.fit_resample(features_np, labels_np)
        return torch.tensor(features_resampled, dtype=torch.float), torch.tensor(labels_resampled, dtype=torch.long)
    except:
        return features, labels

def train_ggnn_phase(ggnn, train_loader, optimizer, epochs, patience):
    ggnn.train()
    best_loss = float('inf')
    patience_counter = 0
    for epoch in range(epochs):
        total_loss = 0
        for batch_data in train_loader:
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            output = ggnn(batch_data.x, batch_data.edge_index, batch_data.batch)
            temp_classifier = nn.Linear(GRAPH_HIDDEN_DIM, NUM_CLASSES).to(device)
            logits = temp_classifier(output)
            loss = nn.CrossEntropyLoss()(logits, batch_data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f'GGNN Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

def train_mlp_phase(mlp, features, labels, optimizer, epochs, patience):
    mlp.train()
    best_loss = float('inf')
    patience_counter = 0
    dataset = torch.utils.data.TensorDataset(features, labels)
    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    for epoch in range(epochs):
        total_loss = 0
        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad()
            logits, anchor_features = mlp(batch_features, return_features=True)
            unique_labels = torch.unique(batch_labels)
            if len(unique_labels) < 2:
                loss = nn.CrossEntropyLoss()(logits, batch_labels)
            else:
                positive_indices = []
                negative_indices = []
                for i in range(len(batch_labels)):
                    same_class_mask = (batch_labels == batch_labels[i]) & (torch.arange(len(batch_labels), device=device) != i)
                    diff_class_mask = batch_labels != batch_labels[i]
                    if same_class_mask.any():
                        positive_indices.append(torch.where(same_class_mask)[0][0].item())
                    else:
                        positive_indices.append(i)
                    if diff_class_mask.any():
                        negative_indices.append(torch.where(diff_class_mask)[0][0].item())
                    else:
                        negative_indices.append((i + 1) % len(batch_labels))
                positive_features = anchor_features[positive_indices]
                negative_features = anchor_features[negative_indices]
                loss = triplet_loss(logits, batch_labels, anchor_features, positive_features, negative_features, TRIPLET_ALPHA, TRIPLET_BETA, TRIPLET_GAMMA)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        print(f'MLP Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

def evaluate_model(ggnn, mlp, test_data_list):
    ggnn.eval()
    mlp.eval()
    y_true = []
    y_pred = []
    y_probs = []
    with torch.no_grad():
        for data in test_data_list:
            batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
            graph_embedding = ggnn(data.x.to(device), data.edge_index.to(device), batch)
            logits = mlp(graph_embedding)
            probs = torch.softmax(logits, dim=1)
            predicted_class = torch.argmax(logits, dim=1)
            y_true.append(data.y.item())
            y_pred.append(predicted_class.item())
            y_probs.append(probs.cpu().numpy())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.vstack(y_probs)
    cm = confusion_matrix(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
    except:
        auc = 0.0
    tp = np.sum((y_true == y_pred) & (y_true != 0))
    fn = np.sum((y_true != y_pred) & (y_true != 0))
    results = {
        'Confusion Matrix': cm,
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn
    }
    return results

start_time = time.time()

train_path = '/Users/akter/fahim/codegraph/traintry1.csv'
test_path = '/Users/akter/fahim/codegraph/testtry1.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)



print(f'Train samples: {len(train_df)}, Test samples: {len(test_df)}')

print('Building Word2Vec model...')
train_tokenized = [str(func).split() for func in train_df['func']]
test_tokenized = [str(func).split() for func in test_df['func']]
word2vec_model = Word2Vec(train_tokenized + test_tokenized, vector_size=100, window=10, min_count=1, workers=4)

print('Preprocessing datasets...')
train_data_list = preprocess_dataset(train_df, word2vec_model)
test_data_list = preprocess_dataset(test_df, word2vec_model)
print(f'Processed train: {len(train_data_list)}, test: {len(test_data_list)}')

input_dim = train_data_list[0].x.size(1)
print(f'Input dimension: {input_dim}')

ggnn = GGNNFeatureExtractor(input_dim=input_dim, hidden_dim=GRAPH_HIDDEN_DIM, num_layers=8, dropout_prob=DROPOUT_PROB).to(device)
mlp = MLPRepresentationLearner(input_dim=GRAPH_HIDDEN_DIM, hidden_sizes=MLP_HIDDEN_SIZES, num_classes=NUM_CLASSES).to(device)

print('\n=== Phase I: GGNN Pre-training ===')
train_loader = DataLoader(train_data_list, batch_size=BATCH_SIZE, shuffle=True)
optimizer_ggnn = optim.Adam(ggnn.parameters(), lr=LEARNING_RATE)
train_ggnn_phase(ggnn, train_loader, optimizer_ggnn, EPOCHS_GGNN, PATIENCE_GGNN)

print('\n=== Extracting GGNN Features ===')
train_features, train_labels = extract_features_batched(ggnn, train_data_list)
print(f'Extracted features shape: {train_features.shape}')

print('\n=== Applying SMOTE Resampling ===')
train_features_balanced, train_labels_balanced = apply_smote(train_features, train_labels)
print(f'Balanced features shape: {train_features_balanced.shape}')

print('\n=== Phase II: MLP Representation Learning ===')
optimizer_mlp = optim.Adam(mlp.parameters(), lr=LEARNING_RATE)
train_mlp_phase(mlp, train_features_balanced, train_labels_balanced, optimizer_mlp, EPOCHS_MLP, PATIENCE_MLP)

print('\n=== Evaluating Model ===')
results = evaluate_model(ggnn, mlp, test_data_list)

end_time = time.time()
execution_time = end_time - start_time

print('\n' + '='*60)
print('FINAL EVALUATION RESULTS')
print('='*60)
print(f"Confusion Matrix:\n{results['Confusion Matrix']}")
print(f"AUC: {results['AUC']:.4f}")
print(f"Accuracy: {results['Accuracy']:.4f}")
print(f"Precision: {results['Precision']:.4f}")
print(f"Recall: {results['Recall']:.4f}")
print(f"F1 Score: {results['F1']:.4f}")
print(f"MCC: {results['MCC']:.4f}")
print(f"Cohen's Kappa: {results['Kappa']:.4f}")
print(f"MSE: {results['MSE']:.4f}")
print(f"MAE: {results['MAE']:.4f}")
print(f"TP: {results['TP']}")
print(f"FN: {results['FN']}")
print('='*60)
print(f'\nTotal Execution Time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)')
print('='*60)

# muvul

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    mean_squared_error, mean_absolute_error
)
from sklearn.preprocessing import LabelEncoder, label_binarize
import re
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f'Device: {device}\n')

LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS_GLOBAL_LOCAL = 60
EPOCHS_FUSION = 10
DROPOUT_RATE = 0.5
GLOBAL_NODES = 300
LOCAL_NODES = 200
FUSION_NODES = 500
VECTOR_DIM = 50
MAX_GLOBAL_LENGTH = 300
MAX_LOCAL_LENGTH = 100

def parse_code_gadgets(source_code):
    if pd.isna(source_code) or source_code == '':
        return []
    tokens = re.findall(r'\b\w+\b|[{}();,=+\-*/<>!&|]', str(source_code))
    gadgets = []
    current_gadget = []
    for token in tokens:
        current_gadget.append(token)
        if len(current_gadget) >= 10 or token in [';', '}', '{']:
            if current_gadget:
                gadgets.append(current_gadget)
                current_gadget = []
    if current_gadget:
        gadgets.append(current_gadget)
    return gadgets if gadgets else [["empty"]]

def parse_code_attention(code_gadgets):
    attention_gadgets = []
    keywords = {'if', 'else', 'for', 'while', 'function', 'return', 'var', 'int', 'main', 'printf', 'scanf'}
    for gadget in code_gadgets:
        attention_tokens = []
        for token in gadget:
            if token.lower() in keywords or any(char in token for char in ['(', ')', '{', '}', '=', '<', '>']):
                attention_tokens.append(token)
        if attention_tokens:
            attention_gadgets.append(attention_tokens)
    return attention_gadgets if attention_gadgets else [["empty"]]

def create_vocabulary(all_tokens):
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for token in set(all_tokens):
        if token not in vocab:
            vocab[token] = len(vocab)
    return vocab

def tokens_to_sequences(token_lists, vocab, max_length):
    sequences = []
    for tokens in token_lists:
        seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        if len(seq) < max_length:
            seq += [vocab['<PAD>']] * (max_length - len(seq))
        else:
            seq = seq[:max_length]
        sequences.append(seq)
    return np.array(sequences)

def process_code_samples(df, func_column):
    func_samples = df[func_column].fillna('').tolist()
    code_gadgets = [parse_code_gadgets(code) for code in func_samples]
    code_attentions = [parse_code_attention(gadgets) for gadgets in code_gadgets]
    all_func_tokens = [token for gadgets in code_gadgets for gadget in gadgets for token in gadget]
    vocab = create_vocabulary(all_func_tokens)
    global_token_lists = [[token for gadget in gadgets for token in gadget] for gadgets in code_gadgets]
    global_sequences = tokens_to_sequences(global_token_lists, vocab, MAX_GLOBAL_LENGTH)
    local_token_lists = [[token for gadget in attention_list for token in gadget] for attention_list in code_attentions]
    local_sequences = tokens_to_sequences(local_token_lists, vocab, MAX_LOCAL_LENGTH)
    return global_sequences, local_sequences, vocab

class CustomAttention(nn.Module):
    def __init__(self, hidden_size):
        super(CustomAttention, self).__init__()
        self.hidden_size = hidden_size
        
    def forward(self, lstm_output):
        scores = torch.matmul(lstm_output, lstm_output.transpose(-2, -1))
        attention_weights = torch.softmax(scores, dim=-1)
        attended_output = torch.matmul(attention_weights, lstm_output)
        return attended_output[:, -1, :]

class GlobalFeatureModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, max_length):
        super(GlobalFeatureModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True, bidirectional=True)
        self.attention = CustomAttention(hidden_size * 2)
        self.projection = nn.Linear(hidden_size * 2, hidden_size)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        attended = self.attention(lstm_out)
        output = self.projection(attended)
        return output

class LocalFeatureModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, max_length):
        super(LocalFeatureModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True, bidirectional=True)
        self.attention = CustomAttention(hidden_size * 2)
        self.projection = nn.Linear(hidden_size * 2, hidden_size)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        attended = self.attention(lstm_out)
        output = self.projection(attended)
        return output

class FeatureFusionModel(nn.Module):
    def __init__(self, global_model, local_model, fusion_nodes, num_classes, dropout_rate):
        super(FeatureFusionModel, self).__init__()
        self.global_model = global_model
        self.local_model = local_model
        for param in self.global_model.parameters():
            param.requires_grad = False
        for param in self.local_model.parameters():
            param.requires_grad = False
        self.fusion_layer = nn.Linear(GLOBAL_NODES + LOCAL_NODES, fusion_nodes)
        self.dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(fusion_nodes, num_classes)
        self.tanh = nn.Tanh()
        
    def forward(self, global_input, local_input):
        global_features = self.global_model(global_input)
        local_features = self.local_model(local_input)
        merged_features = torch.cat([global_features, local_features], dim=1)
        fused = self.tanh(self.fusion_layer(merged_features))
        fused = self.dropout(fused)
        output = self.output_layer(fused)
        return output

class VulnerabilityDataset(Dataset):
    def __init__(self, global_sequences, local_sequences, labels=None):
        self.global_sequences = global_sequences
        self.local_sequences = local_sequences
        self.labels = labels
        
    def __len__(self):
        return len(self.global_sequences)
    
    def __getitem__(self, idx):
        if self.labels is not None:
            return (
                torch.tensor(self.global_sequences[idx], dtype=torch.long),
                torch.tensor(self.local_sequences[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long)
            )
        else:
            return (
                torch.tensor(self.global_sequences[idx], dtype=torch.long),
                torch.tensor(self.local_sequences[idx], dtype=torch.long)
            )

def calculate_metrics(y_true, y_pred, y_pred_proba, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    y_true_bin = label_binarize(y_true, classes=range(num_classes))
    if num_classes == 2:
        auc = roc_auc_score(y_true, y_pred_proba[:, 1])
    else:
        auc = roc_auc_score(y_true_bin, y_pred_proba, multi_class='ovr', average='weighted')
    
    tp = 0
    fn = 0
    for i in range(num_classes):
        if i < len(cm):
            tp += cm[i, i] if i < cm.shape[1] else 0
            fn += sum(cm[i, :]) - (cm[i, i] if i < cm.shape[1] else 0)
    
    return {
        'AUC': auc, 'Accuracy': accuracy, 'Precision': precision,
        'Recall': recall, 'F1': f1, 'MCC': mcc, 'Kappa': kappa,
        'MSE': mse, 'MAE': mae, 'TP': tp, 'FN': fn
    }

def train_individual_model(model, train_loader, criterion, optimizer, epochs, model_name, num_classes):
    print(f"\nTraining {model_name}...")
    model.train()
    classifier = nn.Linear(GLOBAL_NODES if 'Global' in model_name else LOCAL_NODES, num_classes).to(device)
    optimizer_with_classifier = optim.RMSprop(list(model.parameters()) + list(classifier.parameters()), lr=LEARNING_RATE)
    
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (global_input, local_input, labels) in enumerate(train_loader):
            input_data = global_input if 'Global' in model_name else local_input
            input_data, labels = input_data.to(device), labels.to(device)
            optimizer_with_classifier.zero_grad()
            features = model(input_data)
            logits = classifier(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer_with_classifier.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

def train_fusion_model(model, train_loader, criterion, optimizer, epochs):
    print(f"\nTraining Fusion Model...")
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (global_input, local_input, labels) in enumerate(train_loader):
            global_input, local_input, labels = global_input.to(device), local_input.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(global_input, local_input)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

def evaluate_model(model, test_loader, num_classes):
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    with torch.no_grad():
        for global_input, local_input, labels in test_loader:
            global_input, local_input = global_input.to(device), local_input.to(device)
            outputs = model(global_input, local_input)
            probabilities = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(outputs, dim=1)
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    metrics = calculate_metrics(all_labels, all_predictions, np.array(all_probabilities), num_classes)
    return metrics

print("Loading datasets...")
train_path = '/Users/akter/fahim/codegraph/traintry1.csv'
test_path = '/Users/akter/fahim/codegraph/testtry1.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)


print(f"Train dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")
print(f"Train label distribution:\n{train_df['label'].value_counts().sort_index()}")
print(f"Test label distribution:\n{test_df['label'].value_counts().sort_index()}")

print("\nProcessing training data...")
X_train_global, X_train_local, vocab = process_code_samples(train_df, "func")

print("\nProcessing test data...")
X_test_global, X_test_local, _ = process_code_samples(test_df, "func")

label_encoder = LabelEncoder()
Y_train = label_encoder.fit_transform(train_df["label"].values)
Y_test = label_encoder.transform(test_df["label"].values)
num_classes = len(label_encoder.classes_)

print(f"\nNumber of classes: {num_classes}")
print(f"Classes: {label_encoder.classes_}")
print(f"Vocabulary size: {len(vocab)}")

train_dataset = VulnerabilityDataset(X_train_global, X_train_local, Y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = VulnerabilityDataset(X_test_global, X_test_local, Y_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

vocab_size = len(vocab)
global_model = GlobalFeatureModel(vocab_size, VECTOR_DIM, GLOBAL_NODES, MAX_GLOBAL_LENGTH).to(device)
local_model = LocalFeatureModel(vocab_size, VECTOR_DIM, LOCAL_NODES, MAX_LOCAL_LENGTH).to(device)

criterion = nn.CrossEntropyLoss()
global_optimizer = optim.RMSprop(global_model.parameters(), lr=LEARNING_RATE)
local_optimizer = optim.RMSprop(local_model.parameters(), lr=LEARNING_RATE)

start_time = time.time()

train_individual_model(global_model, train_loader, criterion, global_optimizer, EPOCHS_GLOBAL_LOCAL, "Global Model", num_classes)
train_individual_model(local_model, train_loader, criterion, local_optimizer, EPOCHS_GLOBAL_LOCAL, "Local Model", num_classes)

print("\nCreating and training fusion model...")
fusion_model = FeatureFusionModel(global_model, local_model, FUSION_NODES, num_classes, DROPOUT_RATE).to(device)
fusion_optimizer = optim.RMSprop(fusion_model.parameters(), lr=LEARNING_RATE)

train_fusion_model(fusion_model, train_loader, criterion, fusion_optimizer, EPOCHS_FUSION)

training_time = time.time() - start_time

print("\n" + "="*80)
print("EVALUATION ON TEST DATASET")
print("="*80)

eval_start = time.time()
metrics = evaluate_model(fusion_model, test_loader, num_classes)
eval_time = time.time() - eval_start

print("\n=== FINAL TEST RESULTS ===")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print(f"\n=== EXECUTION TIME ===")
print(f"Training Time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Evaluation Time: {eval_time:.2f} seconds")
print(f"Total Time: {training_time + eval_time:.2f} seconds ({(training_time + eval_time)/60:.2f} minutes)")
print("="*80)

# MANDO

In [ ]:
import re
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData, Batch
from torch_geometric.nn import GATConv
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, mean_squared_error, mean_absolute_error
)
from sklearn.preprocessing import label_binarize

def remo(code):
    if not isinstance(code, str):
        return code
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'//.*?$', '', code, flags=re.MULTILINE)
    code = re.sub(r'^\s*[\n\r]', '', code, flags=re.MULTILINE)
    return code.strip()

def generate_heterogeneous_graph(code_sample, num_lines=10, num_funcs=5):
    data = HeteroData()
    data['line'].x = torch.randn((num_lines, 64))
    data['func'].x = torch.randn((num_funcs, 64))
    data['line', 'NEXT', 'line'].edge_index = torch.tensor(
        [[i for i in range(num_lines-1)], 
         [i+1 for i in range(num_lines-1)]], dtype=torch.long
    )
    data['func', 'CALLS', 'func'].edge_index = torch.tensor(
        [[i for i in range(num_funcs-1)], 
         [i+1 for i in range(num_funcs-1)]], dtype=torch.long
    )
    data['func', 'CONTAINS', 'line'].edge_index = torch.tensor(
        [[i % num_funcs for i in range(min(num_lines, num_funcs*2))], 
         [i % num_lines for i in range(min(num_lines, num_funcs*2))]], dtype=torch.long
    )
    data['line', 'NEXT_REV', 'line'].edge_index = data['line', 'NEXT', 'line'].edge_index.flip(0)
    data['func', 'CALLS_REV', 'func'].edge_index = data['func', 'CALLS', 'func'].edge_index.flip(0)
    data['line', 'CONTAINS_REV', 'func'].edge_index = data['func', 'CONTAINS', 'line'].edge_index.flip(0)
    return data

class TopologicalGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(TopologicalGNN, self).__init__()
        self.conv1_line = GATConv(in_channels, hidden_channels, heads=4, concat=True)
        self.conv2_line = GATConv(hidden_channels*4, out_channels, heads=1, concat=False)
        self.conv1_func = GATConv(in_channels, hidden_channels, heads=4, concat=True)
        self.conv2_func = GATConv(hidden_channels*4, out_channels, heads=1, concat=False)

    def forward(self, data):
        x_line = data['line'].x
        x_func = data['func'].x
        edge_index_line = data['line', 'NEXT', 'line'].edge_index
        edge_index_func = data['func', 'CALLS', 'func'].edge_index
        x_line = F.elu(self.conv1_line(x_line, edge_index_line))
        x_line = self.conv2_line(x_line, edge_index_line)
        x_func = F.elu(self.conv1_func(x_func, edge_index_func))
        x_func = self.conv2_func(x_func, edge_index_func)
        return {'line': x_line, 'func': x_func}

class MetapathAttention(nn.Module):
    def __init__(self, in_channels, out_channels, num_heads=8):
        super(MetapathAttention, self).__init__()
        self.num_heads = num_heads
        self.out_channels = out_channels
        self.head_dim = out_channels // num_heads
        self.W = nn.Linear(in_channels, out_channels)
        self.attn = nn.Linear(out_channels * 2, 1)
        self.gat = GATConv(in_channels, self.head_dim, heads=num_heads, concat=True)

    def forward(self, x, edge_index):
        x_transformed = self.W(x)
        x_gat = self.gat(x, edge_index)
        src, dst = edge_index
        x_src = x_transformed[src]
        x_dst = x_transformed[dst]
        attn_input = torch.cat([x_src, x_dst], dim=-1)
        attn_scores = torch.sigmoid(self.attn(attn_input))
        aggregated = torch.zeros_like(x_transformed)
        aggregated.index_add_(0, dst, attn_scores * x_src)
        return aggregated + x_gat

class NodeLevelAttentionHGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_heads=8):
        super(NodeLevelAttentionHGNN, self).__init__()
        self.line_attention_next = MetapathAttention(in_channels, hidden_channels, num_heads)
        self.line_attention_next_rev = MetapathAttention(in_channels, hidden_channels, num_heads)
        self.func_attention_calls = MetapathAttention(in_channels, hidden_channels, num_heads)
        self.func_attention_calls_rev = MetapathAttention(in_channels, hidden_channels, num_heads)
        self.cross_attention = MetapathAttention(in_channels, hidden_channels, num_heads)
        self.fc = nn.Linear(hidden_channels * 5, out_channels)

    def forward(self, x_dict, data):
        x_line = x_dict['line']
        x_func = x_dict['func']
        line_emb_next = self.line_attention_next(x_line, data['line', 'NEXT', 'line'].edge_index)
        line_emb_next_rev = self.line_attention_next_rev(x_line, data['line', 'NEXT_REV', 'line'].edge_index)
        func_emb_calls = self.func_attention_calls(x_func, data['func', 'CALLS', 'func'].edge_index)
        func_emb_calls_rev = self.func_attention_calls_rev(x_func, data['func', 'CALLS_REV', 'func'].edge_index)
        cross_edge_index = data['func', 'CONTAINS', 'line'].edge_index
        x_combined = torch.cat([x_func, torch.zeros(x_line.size(0) - x_func.size(0), x_func.size(1)).to(x_func.device)])
        cross_emb = self.cross_attention(x_combined[:x_line.size(0)], cross_edge_index)
        line_final = torch.cat([line_emb_next, line_emb_next_rev, cross_emb[:x_line.size(0)]], dim=-1)
        func_final = torch.cat([func_emb_calls, func_emb_calls_rev], dim=-1)
        combined_line = self.fc(torch.cat([line_final, torch.zeros(x_line.size(0), func_final.size(1)).to(line_final.device)], dim=-1))
        combined_func = self.fc(torch.cat([torch.zeros(x_func.size(0), line_final.size(1)).to(func_final.device), func_final], dim=-1))
        return {'line': combined_line, 'func': combined_func}

class MANDOClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout_rate=0.5):
        super(MANDOClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class MANDOFramework(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_classes, num_heads=8):
        super(MANDOFramework, self).__init__()
        self.topological_gnn = TopologicalGNN(in_channels, hidden_channels, out_channels)
        self.node_attention_hgnn = NodeLevelAttentionHGNN(out_channels, hidden_channels, out_channels, num_heads)
        self.classifier = MANDOClassifier(out_channels, hidden_channels, num_classes)

    def forward(self, data):
        topo_embeddings = self.topological_gnn(data)
        node_embeddings = self.node_attention_hgnn(topo_embeddings, data)
        func_emb_mean = torch.mean(node_embeddings['func'], dim=0)
        line_emb_mean = torch.mean(node_embeddings['line'], dim=0)
        graph_embedding = (func_emb_mean + line_emb_mean) / 2
        logits = self.classifier(graph_embedding)
        return logits

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f'Device: {device}\n')

train_path = '/Users/akter/fahim/codegraph/traintry1.csv'
test_path = '/Users/akter/fahim/codegraph/testtry1.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)


train_df['func'] = train_df['func'].apply(remo)
test_df['func'] = test_df['func'].apply(remo)

start_time = time.time()

train_graphs = []
for _, row in train_df.iterrows():
    code_sample = str(row['func'])
    graph_data = generate_heterogeneous_graph(code_sample)
    train_graphs.append(graph_data)

test_graphs = []
for _, row in test_df.iterrows():
    code_sample = str(row['func'])
    graph_data = generate_heterogeneous_graph(code_sample)
    test_graphs.append(graph_data)

y_train = torch.tensor(train_df['label'].values, dtype=torch.long)
y_test = torch.tensor(test_df['label'].values, dtype=torch.long)

in_channels = 64
hidden_channels = 128
out_channels = 128
num_classes = 6
num_heads = 8
learning_rate = 0.001
epochs = 50
dropout_rate = 0.5

model = MANDOFramework(in_channels, hidden_channels, out_channels, num_classes, num_heads).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

model.train()
for epoch in range(epochs):
    total_loss = 0
    for i, graph_data in enumerate(train_graphs):
        graph_data = graph_data.to(device)
        optimizer.zero_grad()
        logits = model(graph_data)
        loss = criterion(logits.unsqueeze(0), y_train[i].unsqueeze(0).to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_graphs):.4f}')

model.eval()
y_pred_list = []
y_prob_list = []
with torch.no_grad():
    for i, graph_data in enumerate(test_graphs):
        graph_data = graph_data.to(device)
        logits = model(graph_data)
        y_prob = F.softmax(logits, dim=0)
        y_pred = torch.argmax(y_prob)
        y_pred_list.append(y_pred.cpu().item())
        y_prob_list.append(y_prob.cpu().numpy())

end_time = time.time()
execution_time = end_time - start_time

y_test_np = y_test.numpy()
y_pred_np = np.array(y_pred_list)
y_prob_np = np.array(y_prob_list)

cm = confusion_matrix(y_test_np, y_pred_np)
accuracy = accuracy_score(y_test_np, y_pred_np)
precision = precision_score(y_test_np, y_pred_np, average='macro', zero_division=0)
recall = recall_score(y_test_np, y_pred_np, average='macro', zero_division=0)
f1 = f1_score(y_test_np, y_pred_np, average='macro', zero_division=0)
mcc = matthews_corrcoef(y_test_np, y_pred_np)
kappa = cohen_kappa_score(y_test_np, y_pred_np)
mse = mean_squared_error(y_test_np, y_pred_np)
mae = mean_absolute_error(y_test_np, y_pred_np)

y_test_bin = label_binarize(y_test_np, classes=range(num_classes))
try:
    auc = roc_auc_score(y_test_bin, y_prob_np, average='macro', multi_class='ovr')
except:
    auc = 0.0

tp = 0
fn = 0
for i in range(len(cm)):
    tp += cm[i, i]
    fn += sum(cm[i, :]) - cm[i, i]

print(f"\nConfusion Matrix:\n{cm}")
print(f"\nPerformance Metrics:")
print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"Kappa: {kappa:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"TP: {tp}")
print(f"FN: {fn}")
print(f"\nTotal Execution Time: {execution_time:.2f} seconds")

# sysver

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import networkx as nx
from gensim.models.word2vec import Word2Vec
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, 
                            recall_score, f1_score, matthews_corrcoef, 
                            cohen_kappa_score, roc_auc_score, mean_squared_error, 
                            mean_absolute_error)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

EMBEDDING_DIM = 40
MAXLEN = 198
MAX_SLICES = 9
BATCH_SIZE = 128
EPOCHS = 10
HIDDEN_DIM = 256
DROPOUT = 0.2
OUTPUT_CLASSES = 6

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}\n')

def remo(s):
    if not isinstance(s, str):
        s = '' if s is None else str(s)
    return s.replace('\r', ' ').replace('\n', ' ').strip()

def build_pdg(source_code):
    pdg = nx.DiGraph()
    statements = source_code.split('\n')
    for i, stmt in enumerate(statements):
        pdg.add_node(i, code=stmt, type="Statement", location=f"{i+1}:0")
    for i in range(len(statements) - 1):
        pdg.add_edge(i, i + 1, type="ControlDependency")
    for i, stmt in enumerate(statements):
        if "=" in stmt:
            var_name = stmt.split("=")[0].strip()
            for j, other_stmt in enumerate(statements):
                if var_name in other_stmt and i != j:
                    pdg.add_edge(i, j, type="DataDependency", var=var_name)
    return pdg

def modify_pdg_nodes(pdg):
    for _, data in pdg.nodes(data=True):
        if data.get('type') == "Statement":
            data['code'] = str(data.get('code', '')).strip()
    return pdg

def extract_slices_from_pdg(pdg):
    slices = []
    for node in pdg.nodes:
        slice_nodes = list(nx.ancestors(pdg, node)) + [node]
        slices.append([pdg.nodes[n].get('code', '') for n in slice_nodes])
    return slices

def process_functions_with_pdg_grouped(df, func_col):
    grouped_slices = []
    for func in df[func_col].tolist():
        pdg = build_pdg(func)
        pdg = modify_pdg_nodes(pdg)
        slices = extract_slices_from_pdg(pdg)
        flattened_slices = [' '.join(slc) for slc in slices]
        grouped_slices.append(flattened_slices)
    return grouped_slices

def train_word2vec(corpus):
    w2v_model = Word2Vec(sentences=corpus, vector_size=EMBEDDING_DIM, 
                        window=5, min_count=1, workers=4, sg=1, epochs=5)
    return w2v_model

def embed_grouped_tokens(grouped_tokens, w2v_model):
    embedded_grouped = []
    for token_group in grouped_tokens:
        embedded_group = []
        for token_list in token_group:
            toks = str(token_list).split()
            vecs = [w2v_model.wv[t] for t in toks if t in w2v_model.wv]
            if len(vecs) > 0:
                embedded_group.append(vecs)
        if not embedded_group:
            embedded_group = [[[0.0] * EMBEDDING_DIM] * MAXLEN]
        embedded_grouped.append(embedded_group)
    return embedded_grouped

def pad_grouped_slices(grouped_vectors, maxlen, embedding_dim, max_slices):
    padded_grouped = []
    for group in grouped_vectors:
        padded_group = []
        for vec in group[:max_slices]:
            if len(vec) > maxlen:
                vec = vec[:maxlen]
            elif len(vec) < maxlen:
                vec = vec + [[0.0] * embedding_dim] * (maxlen - len(vec))
            padded_group.append(vec)
        while len(padded_group) < max_slices:
            padded_group.append([[0.0] * embedding_dim] * maxlen)
        padded_grouped.append(padded_group)
    return np.array(padded_grouped, dtype=np.float32)

class VulnerabilityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class BGRUModel(nn.Module):
    def __init__(self, input_dim, seq_len, hidden_dim, output_dim, dropout):
        super(BGRUModel, self).__init__()
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, 
                         bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
    
    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, self.seq_len, self.input_dim)
        _, hidden = self.gru(x)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.fc(hidden)
        return out

def compute_metrics(y_true, y_pred, y_prob, num_classes):
    metrics = {}
    y_true_np = y_true.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()
    y_prob_np = y_prob.cpu().numpy()
    
    metrics['Accuracy'] = accuracy_score(y_true_np, y_pred_np)
    metrics['Precision'] = precision_score(y_true_np, y_pred_np, 
                                          average='weighted', zero_division=0)
    metrics['Recall'] = recall_score(y_true_np, y_pred_np, 
                                    average='weighted', zero_division=0)
    metrics['F1'] = f1_score(y_true_np, y_pred_np, 
                            average='weighted', zero_division=0)
    metrics['MCC'] = matthews_corrcoef(y_true_np, y_pred_np)
    metrics['Kappa'] = cohen_kappa_score(y_true_np, y_pred_np)
    
    try:
        metrics['AUC'] = roc_auc_score(y_true_np, y_prob_np, 
                                      multi_class='ovr', average='weighted')
    except:
        metrics['AUC'] = 0.0
    
    metrics['MSE'] = mean_squared_error(y_true_np, y_pred_np)
    metrics['MAE'] = mean_absolute_error(y_true_np, y_pred_np)
    
    cm = confusion_matrix(y_true_np, y_pred_np)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    metrics['TP'] = int(tp)
    metrics['FN'] = int(fn)
    
    return metrics

def main():
    start_time = time.time()
    
    print("Loading datasets...")
    train_path = '/home/m/mdfahimsultan/codegraph/trainpro (1).csv'
    test_path = '/home/m/mdfahimsultan/codegraph/testpro.csv'
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
   
    

    
    print("Extracting program slices using PDG...")
    train_grouped_slices = process_functions_with_pdg_grouped(train_df, 'func')
    test_grouped_slices = process_functions_with_pdg_grouped(test_df, 'func')
    
    print("Training Word2Vec model...")
    train_sentences = [str(slc).split() for group in train_grouped_slices for slc in group]
    if len(train_sentences) == 0:
        train_sentences = [["__pad__"]]
    w2v_model = train_word2vec(train_sentences)
    
    print("Embedding tokens...")
    train_grouped_vectors = embed_grouped_tokens(train_grouped_slices, w2v_model)
    test_grouped_vectors = embed_grouped_tokens(test_grouped_slices, w2v_model)
    
    print("Padding sequences...")
    train_padded = pad_grouped_slices(train_grouped_vectors, MAXLEN, 
                                     EMBEDDING_DIM, MAX_SLICES)
    test_padded = pad_grouped_slices(test_grouped_vectors, MAXLEN, 
                                    EMBEDDING_DIM, MAX_SLICES)
    
    seq_len = train_padded.shape[1] * train_padded.shape[2]
    input_dim = train_padded.shape[-1]
    
    train_seq = train_padded.reshape(train_padded.shape[0], seq_len, input_dim)
    test_seq = test_padded.reshape(test_padded.shape[0], seq_len, input_dim)
    
    y_train = train_df['label'].astype(int).values
    y_test = test_df['label'].astype(int).values
    
    train_dataset = VulnerabilityDataset(train_seq, y_train)
    test_dataset = VulnerabilityDataset(test_seq, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"Building BGRU model on {device}...")
    model = BGRUModel(input_dim, seq_len, HIDDEN_DIM, OUTPUT_CLASSES, DROPOUT).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    print("Training model...")
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")
    
    print("\nEvaluating on test set...")
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(batch_y)
            all_probs.append(probs.cpu())
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_probs = torch.cat(all_probs)
    
    metrics = compute_metrics(all_labels, all_preds, all_probs, OUTPUT_CLASSES)
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    print("\n" + "="*80)
    print("FINAL TEST RESULTS")
    print("="*80)
    for metric_name, metric_value in metrics.items():
        if isinstance(metric_value, float):
            print(f"{metric_name}: {metric_value:.4f}")
        else:
            print(f"{metric_name}: {metric_value}")
    print("="*80)
    print(f"\nTotal Execution Time: {execution_time:.2f} seconds")
    print(f"Total Execution Time: {execution_time/60:.2f} minutes")
    print("="*80)

if __name__ == "__main__":
    main()

# bgggn4

In [ ]:
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data, DataLoader as PyGDataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, 
    mean_absolute_error, confusion_matrix, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import label_binarize
import re
from tqdm import tqdm
import time
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f'Device: {device}\n')

def remove_comments(code):
    if not isinstance(code, str):
        return code
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'//.*?$', '', code, flags=re.MULTILINE)
    code = re.sub(r'^\s*[\n\r]', '', code, flags=re.MULTILINE)
    return code.strip()

def generate_ast(code):
    ast = nx.DiGraph()
    tokens = code.split()
    ast.add_node("root", token="root", node_type="root")
    for i, token in enumerate(tokens[:min(20, len(tokens))]):
        node_id = f"ast_{i}"
        ast.add_node(node_id, token=token, node_type="token")
        ast.add_edge("root", node_id)
    return ast

def generate_cfg(code):
    cfg = nx.DiGraph()
    lines = [l.strip() for l in code.split('\n') if l.strip()]
    cfg.add_node("start", token="start", node_type="control")
    prev = "start"
    for i, line in enumerate(lines[:min(15, len(lines))]):
        node_id = f"cfg_{i}"
        cfg.add_node(node_id, token=line[:30], node_type="statement")
        cfg.add_edge(prev, node_id)
        prev = node_id
    cfg.add_node("end", token="end", node_type="control")
    cfg.add_edge(prev, "end")
    return cfg

def generate_dfg(code):
    dfg = nx.DiGraph()
    vars_found = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code)
    unique_vars = list(set(vars_found))[:min(10, len(set(vars_found)))]
    for i, var in enumerate(unique_vars):
        node_id = f"dfg_{i}"
        dfg.add_node(node_id, token=var, node_type="variable")
    for i in range(len(unique_vars) - 1):
        dfg.add_edge(f"dfg_{i}", f"dfg_{i+1}")
    return dfg

def combine_graphs(ast, cfg, dfg):
    ccg = nx.DiGraph()
    for g in [ast, cfg, dfg]:
        for node, data in g.nodes(data=True):
            ccg.add_node(node, **data)
        for u, v in g.edges():
            ccg.add_edge(u, v)
    return ccg

def train_word2vec_model(func_list):
    sentences = []
    for func in func_list:
        if isinstance(func, str):
            tokens = re.findall(r'\b\w+\b', func.lower())
            if tokens:
                sentences.append(tokens)
    if not sentences:
        sentences = [['default', 'token']]
    model = Word2Vec(sentences, vector_size=50, window=5, min_count=1, workers=4, epochs=10)
    return model

def encode_node_features(node_token, node_type, word2vec_model, type_encoder):
    tokens = re.findall(r'\b\w+\b', str(node_token).lower())
    if tokens:
        token_vectors = [word2vec_model.wv[t] if t in word2vec_model.wv else np.zeros(50) for t in tokens]
        token_emb = np.mean(token_vectors, axis=0)
    else:
        token_emb = np.zeros(50)
    type_encoding = type_encoder.transform([node_type])[0] if node_type in type_encoder.classes_ else 0
    type_vec = np.array([type_encoding], dtype=np.float32)
    return np.concatenate([token_emb, type_vec])

def convert_ccg_to_pyg_data(ccg, word2vec_model, type_encoder, label):
    if len(ccg.nodes()) == 0:
        x = torch.zeros((1, 51), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        y = torch.tensor([label], dtype=torch.long)
        return Data(x=x, edge_index=edge_index, y=y)
    
    node_mapping = {node: idx for idx, node in enumerate(ccg.nodes())}
    node_features = []
    for node in ccg.nodes():
        node_data = ccg.nodes[node]
        token = node_data.get('token', node)
        node_type = node_data.get('node_type', 'unknown')
        feat = encode_node_features(token, node_type, word2vec_model, type_encoder)
        node_features.append(feat)
    
    x = torch.tensor(node_features, dtype=torch.float)
    
    edge_list = []
    for u, v in ccg.edges():
        edge_list.append([node_mapping[u], node_mapping[v]])
        edge_list.append([node_mapping[v], node_mapping[u]])
    
    if edge_list:
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    
    y = torch.tensor([label], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)

class GRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(GRUCell, self).__init__()
        self.hidden_size = hidden_size
        self.linear_z = nn.Linear(input_size + hidden_size, hidden_size)
        self.linear_r = nn.Linear(input_size + hidden_size, hidden_size)
        self.linear_h = nn.Linear(input_size + hidden_size, hidden_size)
    
    def forward(self, x, h):
        combined = torch.cat([x, h], dim=1)
        z = torch.sigmoid(self.linear_z(combined))
        r = torch.sigmoid(self.linear_r(combined))
        combined_r = torch.cat([x, r * h], dim=1)
        h_tilde = torch.tanh(self.linear_h(combined_r))
        h_new = (1 - z) * h + z * h_tilde
        return h_new

class BGNNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(BGNNLayer, self).__init__(aggr='add')
        self.lin = nn.Linear(in_channels, out_channels)
        self.gru = GRUCell(out_channels, out_channels)
    
    def forward(self, x, edge_index):
        h = torch.zeros(x.size(0), self.gru.hidden_size, device=x.device)
        if x.size(1) != self.lin.in_features:
            x = F.pad(x, (0, self.lin.in_features - x.size(1)))
        for _ in range(8):
            m = self.propagate(edge_index, x=x, h=h)
            h = self.gru(m, h)
        return h
    
    def message(self, x_j, h_j):
        return self.lin(x_j)

class BGNN4VD(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes, dropout=0.2):
        super(BGNN4VD, self).__init__()
        self.bgnn = BGNNLayer(in_channels, hidden_channels)
        self.conv1d = nn.Conv1d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_channels, num_classes)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.bgnn(x, edge_index)
        
        node_counts = torch.bincount(batch)
        max_nodes = node_counts.max().item()
        batch_size = node_counts.size(0)
        
        h_padded = torch.zeros(batch_size, max_nodes, h.size(1), device=h.device)
        for i in range(batch_size):
            mask = batch == i
            nodes_i = h[mask]
            h_padded[i, :nodes_i.size(0), :] = nodes_i
        
        h_t = h_padded.transpose(1, 2)
        h_conv = F.relu(self.conv1d(h_t))
        h_pool = self.pool(h_conv).squeeze(-1)
        h_drop = self.dropout(h_pool)
        out = self.fc(h_drop)
        return out

def train_model(model, train_loader, val_loader, epochs, lr):
    optimizer = torch.optim.Adamax(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data)
                pred = out.argmax(dim=1)
                correct += (pred == data.y).sum().item()
                total += data.y.size(0)
        val_acc = correct / total if total > 0 else 0
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}")

def evaluate_model(model, test_loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for data in tqdm(test_loader, desc="Evaluating"):
            data = data.to(device)
            out = model(data)
            probs = F.softmax(out, dim=1)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    
    cm = confusion_matrix(all_labels, all_preds)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    
    try:
        y_bin = label_binarize(all_labels, classes=list(range(num_classes)))
        if num_classes == 2:
            auc = roc_auc_score(all_labels, all_probs[:, 1])
        else:
            auc = roc_auc_score(y_bin, all_probs, average='weighted', multi_class='ovr')
    except:
        auc = 0.0
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': int(tp),
        'FN': int(fn)
    }

def main():
    start_time = time.time()
    
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
    

    
    num_classes = len(train_df['label'].unique())
    print(f"Number of classes: {num_classes}")
    
    print("Training Word2Vec model...")
    all_funcs = pd.concat([train_df['func'], test_df['func']]).tolist()
    word2vec_model = train_word2vec_model(all_funcs)
    
    print("Creating type encoder...")
    all_types = ['root', 'token', 'control', 'statement', 'variable', 'unknown']
    type_encoder = LabelEncoder()
    type_encoder.fit(all_types)
    
    print("Generating CCGs for training data...")
    train_graphs = []
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train CCGs"):
        code = row['func']
        label = row['label']
        ast = generate_ast(code)
        cfg = generate_cfg(code)
        dfg = generate_dfg(code)
        ccg = combine_graphs(ast, cfg, dfg)
        pyg_data = convert_ccg_to_pyg_data(ccg, word2vec_model, type_encoder, label)
        train_graphs.append(pyg_data)
    
    print("Generating CCGs for test data...")
    test_graphs = []
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test CCGs"):
        code = row['func']
        label = row['label']
        ast = generate_ast(code)
        cfg = generate_cfg(code)
        dfg = generate_dfg(code)
        ccg = combine_graphs(ast, cfg, dfg)
        pyg_data = convert_ccg_to_pyg_data(ccg, word2vec_model, type_encoder, label)
        test_graphs.append(pyg_data)
    
    print("Splitting training data into train/val...")
    train_graphs, val_graphs = train_test_split(train_graphs, test_size=0.1, random_state=42)
    
    train_loader = PyGDataLoader(train_graphs, batch_size=32, shuffle=True)
    val_loader = PyGDataLoader(val_graphs, batch_size=32, shuffle=False)
    test_loader = PyGDataLoader(test_graphs, batch_size=32, shuffle=False)
    
    print("Initializing BGNN4VD model...")
    model = BGNN4VD(in_channels=51, hidden_channels=200, num_classes=num_classes, dropout=0.2)
    
    print("Training model...")
    train_model(model, train_loader, val_loader, epochs=5, lr=0.00015)
    
    print("\nEvaluating on test set...")
    results = evaluate_model(model, test_loader, num_classes)
    
    print("\n" + "="*60)
    print("FINAL TEST RESULTS")
    print("="*60)
    for metric, value in results.items():
        if isinstance(value, float):
            print(f"{metric}: {value:.4f}")
        else:
            print(f"{metric}: {value}")
    
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"\nTotal Execution Time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")
    
    torch.save(model.state_dict(), 'bgnn4vd_model.pth')
    print("\nModel saved as 'bgnn4vd_model.pth'")

if __name__ == "__main__":
    main()

Device: mps

Loading datasets...
Balancing training data...
Balancing test data...
Train size: 21793, Test size: 5400
Cleaning code...
Number of classes: 6
Training Word2Vec model...
Creating type encoder...
Generating CCGs for training data...


Train CCGs: 100%|████████████████████████| 21793/21793 [00:42<00:00, 509.62it/s]


Generating CCGs for test data...


Test CCGs: 100%|███████████████████████████| 5400/5400 [00:10<00:00, 515.67it/s]


Splitting training data into train/val...
Initializing BGNN4VD model...
Training model...


...

Epoch 4/5 - Loss: 0.9677, Val Acc: 0.6261


Epoch 5/5: 100%|██████████████████████████████| 613/613 [00:48<00:00, 12.56it/s]


Epoch 5/5 - Loss: 0.9383, Val Acc: 0.6353

Evaluating on test set...


Evaluating: 100%|█████████████████████████████| 169/169 [00:08<00:00, 19.08it/s]


FINAL TEST RESULTS
AUC: 0.8927
Accuracy: 0.6215
Precision: 0.6374
Recall: 0.6215
F1: 0.6248
MCC: 0.5477
Kappa: 0.5456
MSE: 2.7519
MAE: 0.9311
TP: 3356
FN: 2044

Total Execution Time: 342.82 seconds (5.71 minutes)

Model saved as 'bgnn4vd_model.pth'


# vlauflowpath

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import re
import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error,
    mean_absolute_error, roc_auc_score, confusion_matrix
)
from sklearn.preprocessing import label_binarize

EMBED_DIM = 128
BATCH_SIZE = 64
EPOCHS_CONTRASTIVE = 20
EPOCHS_DETECTION = 20
LEARNING_RATE = 0.002
NUM_CLASSES = 6
MAX_SEQ_LEN = 50
NUM_PATHS = 100
SAMPLING_RATIO = 0.3

def clean_code(code):
    if not isinstance(code, str):
        return code
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'//.*?$', '', code, flags=re.MULTILINE)
    code = re.sub(r'^\s*[\n\r]', '', code, flags=re.MULTILINE)
    return code.strip()

def extract_value_flow_paths(code_snippet, num_paths=NUM_PATHS):
    tokens = code_snippet.split()
    paths = []
    if len(tokens) < 3:
        tokens = tokens * 3
    for _ in range(num_paths):
        path_len = np.random.randint(3, min(10, len(tokens) + 1))
        start_idx = np.random.randint(0, max(1, len(tokens) - path_len + 1))
        path = tokens[start_idx:start_idx + path_len]
        paths.append(' '.join(path))
    return paths

def tokenize_path(path, max_seq_len=MAX_SEQ_LEN):
    tokens = np.random.rand(max_seq_len, EMBED_DIM)
    return torch.tensor(tokens, dtype=torch.float)

class CodeDataset(Dataset):
    def __init__(self, dataframe):
        self.funcs = dataframe['func'].tolist()
        self.labels = dataframe['label'].tolist()
    
    def __len__(self):
        return len(self.funcs)
    
    def __getitem__(self, idx):
        return self.funcs[idx], self.labels[idx]

class StatementEncoder(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.node_type_embed = nn.Embedding(100, embed_dim)
        self.node_token_embed = nn.Embedding(1000, embed_dim)
        self.attention = nn.Linear(embed_dim, 1)
        self.fc = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        node_features = x
        alpha = torch.softmax(self.attention(node_features), dim=1)
        pooled = (alpha * node_features).sum(dim=1)
        out = self.dropout(self.fc(pooled))
        return out

class ValueFlowPathEncoder(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.statement_encoder = StatementEncoder(embed_dim)
        self.local_encoder = nn.LSTM(embed_dim, embed_dim, bidirectional=True, batch_first=True)
        self.global_encoder = nn.GRU(embed_dim * 2, embed_dim, batch_first=True)
        self.attention = nn.Linear(embed_dim, 1)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, path_tokens):

        batch_size, num_paths, num_statements, seq_len, embed_dim = path_tokens.shape
        path_tokens = path_tokens.view(batch_size * num_paths, num_statements, seq_len, embed_dim)


        path_tokens = path_tokens.view(-1, seq_len, embed_dim)
        statement_features = self.statement_encoder(path_tokens)
        statement_features = statement_features.view(batch_size * num_paths, num_statements, -1)

        local_features, _ = self.local_encoder(statement_features)
        global_features, _ = self.global_encoder(local_features)
        attn_weights = torch.softmax(self.attention(global_features), dim=1)
        context = (attn_weights * global_features).sum(dim=1)


        context = context.view(batch_size, num_paths, -1)
        context = context.mean(dim=1) 
        return self.dropout(context)


class TransformerDetector(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, num_classes):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attention = nn.Linear(embed_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim // 2, num_classes)
        )
    
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        features = self.transformer(x)
        attn_weights = torch.softmax(self.attention(features), dim=1)
        context = (attn_weights * features).sum(dim=1)
        return self.classifier(context)

def contrastive_loss_fn(z_i, z_j, temperature=0.5):
    batch_size = z_i.size(0)
    z_i = nn.functional.normalize(z_i, dim=1)
    z_j = nn.functional.normalize(z_j, dim=1)
    representations = torch.cat([z_i, z_j], dim=0)
    similarity_matrix = torch.matmul(representations, representations.T)
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z_i.device)
    similarity_matrix = similarity_matrix.masked_fill(mask, -9e15)
    positives = torch.cat([torch.diag(similarity_matrix, batch_size), 
                          torch.diag(similarity_matrix, -batch_size)], dim=0)
    nominator = torch.exp(positives / temperature)
    denominator = torch.sum(torch.exp(similarity_matrix / temperature), dim=1)
    loss = -torch.mean(torch.log(nominator / denominator))
    return loss

def active_learning_path_selection(path_embeddings, sampling_ratio=SAMPLING_RATIO):
    num_paths = path_embeddings.size(0)
    num_select = max(1, int(num_paths * sampling_ratio))
    norms = torch.norm(path_embeddings, p=2, dim=1)
    _, indices = torch.topk(norms, num_select)
    return indices

def path_feasibility_check(paths):
    feasible_paths = []
    for path in paths:
        if len(path) > 0:
            feasible_paths.append(path)
    return feasible_paths if feasible_paths else paths

def train_contrastive(encoder, train_loader, device, epochs):
    optimizer = optim.Adam(encoder.parameters(), lr=LEARNING_RATE)
    encoder.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for funcs, _ in train_loader:
            batch_paths = []
            for func in funcs:
                paths = extract_value_flow_paths(func)
                paths = path_feasibility_check(paths)
                batch_paths.append(paths)
            
            path_tokens_list = []
            for paths in batch_paths:
                path_tokens = []
                for path in paths[:10]:
                    statements = path.split()[:5]
                    statement_tokens = []
                    for _ in statements:
                        statement_tokens.append(tokenize_path(path))
                    if len(statement_tokens) < 5:
                        statement_tokens.extend([tokenize_path(path)] * (5 - len(statement_tokens)))
                    path_tokens.append(torch.stack(statement_tokens))
                if len(path_tokens) < 10:
                    path_tokens.extend([torch.stack([tokenize_path(paths[0])] * 5)] * (10 - len(path_tokens)))
                path_tokens_list.append(torch.stack(path_tokens))
            
            path_tokens_batch = torch.stack(path_tokens_list).to(device)
            
            z_i = encoder(path_tokens_batch)
            z_j = encoder(path_tokens_batch)
            
            loss = contrastive_loss_fn(z_i, z_j)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f'Contrastive Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}')

def train_detector(encoder, detector, train_loader, device, epochs):
    encoder.eval()
    detector.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(detector.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(epochs):
        total_loss = 0
        for funcs, labels in train_loader:
            batch_paths = []
            for func in funcs:
                paths = extract_value_flow_paths(func)
                paths = path_feasibility_check(paths)
                batch_paths.append(paths)
            
            path_tokens_list = []
            for paths in batch_paths:
                path_tokens = []
                for path in paths[:10]:
                    statements = path.split()[:5]
                    statement_tokens = []
                    for _ in statements:
                        statement_tokens.append(tokenize_path(path))
                    if len(statement_tokens) < 5:
                        statement_tokens.extend([tokenize_path(path)] * (5 - len(statement_tokens)))
                    path_tokens.append(torch.stack(statement_tokens))
                if len(path_tokens) < 10:
                    path_tokens.extend([torch.stack([tokenize_path(paths[0])] * 5)] * (10 - len(path_tokens)))
                path_tokens_list.append(torch.stack(path_tokens))
            
            path_tokens_batch = torch.stack(path_tokens_list).to(device)
            labels = torch.tensor(labels, dtype=torch.long).to(device)
            
            with torch.no_grad():
                path_embeddings = encoder(path_tokens_batch)
            
            selected_indices = active_learning_path_selection(path_embeddings)
            selected_embeddings = path_embeddings[selected_indices]
            
            if selected_embeddings.size(0) < path_embeddings.size(0):
                selected_embeddings = torch.cat([selected_embeddings, 
                                                path_embeddings[:path_embeddings.size(0) - selected_embeddings.size(0)]])
            
            logits = detector(selected_embeddings)
            loss = criterion(logits, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f'Detection Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}')

def evaluate(encoder, detector, test_loader, device, num_classes):
    encoder.eval()
    detector.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for funcs, labels in test_loader:
            batch_paths = []
            for func in funcs:
                paths = extract_value_flow_paths(func)
                paths = path_feasibility_check(paths)
                batch_paths.append(paths)
            
            path_tokens_list = []
            for paths in batch_paths:
                path_tokens = []
                for path in paths[:10]:
                    statements = path.split()[:5]
                    statement_tokens = []
                    for _ in statements:
                        statement_tokens.append(tokenize_path(path))
                    if len(statement_tokens) < 5:
                        statement_tokens.extend([tokenize_path(path)] * (5 - len(statement_tokens)))
                    path_tokens.append(torch.stack(statement_tokens))
                if len(path_tokens) < 10:
                    path_tokens.extend([torch.stack([tokenize_path(paths[0])] * 5)] * (10 - len(path_tokens)))
                path_tokens_list.append(torch.stack(path_tokens))
            
            path_tokens_batch = torch.stack(path_tokens_list).to(device)
            
            path_embeddings = encoder(path_tokens_batch)
            selected_indices = active_learning_path_selection(path_embeddings)
            selected_embeddings = path_embeddings[selected_indices]
            
            if selected_embeddings.size(0) < path_embeddings.size(0):
                selected_embeddings = torch.cat([selected_embeddings, 
                                                path_embeddings[:path_embeddings.size(0) - selected_embeddings.size(0)]])
            
            logits = detector(selected_embeddings)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    
    y_bin = label_binarize(all_labels, classes=range(num_classes))
    if num_classes == 2:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    else:
        auc = roc_auc_score(y_bin, all_probs, average='weighted', multi_class='ovr')
    
    cm = confusion_matrix(all_labels, all_preds)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': int(tp),
        'FN': int(fn)
    }

def main():
    start_time = time.time()
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    

    
    train_df['func'] = train_df['func'].apply(clean_code)
    test_df['func'] = test_df['func'].apply(clean_code)
    
    train_dataset = CodeDataset(train_df)
    test_dataset = CodeDataset(test_df)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print("Phase (a): Contrastive Value-Flow Embedding")
    encoder = ValueFlowPathEncoder(EMBED_DIM).to(device)
    train_contrastive(encoder, train_loader, device, EPOCHS_CONTRASTIVE)
    
    print("\nPhase (b): Value-Flow Path Selection (integrated in training)")
    
    print("\nPhase (c): Detection Model Training")
    detector = TransformerDetector(EMBED_DIM, num_heads=8, num_layers=2, num_classes=NUM_CLASSES).to(device)
    train_detector(encoder, detector, train_loader, device, EPOCHS_DETECTION)
    
    print("\n" + "="*60)
    print("Evaluation on Test Set")
    print("="*60)
    results = evaluate(encoder, detector, test_loader, device, NUM_CLASSES)
    
    for metric, value in results.items():
        if isinstance(value, float):
            print(f"{metric}: {value:.4f}")
        else:
            print(f"{metric}: {value}")
    
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"\nTotal Execution Time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")

if __name__ == "__main__":
    main()

# devign

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    confusion_matrix, mean_squared_error, mean_absolute_error
)
from sklearn.preprocessing import label_binarize
import time
import re
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

class CompositeGraphBuilder:
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
        self.token_to_idx = {}
        self.idx_to_token = {}
        self.current_idx = 0
        
    def tokenize_code(self, code):
        code = str(code).lower()
        tokens = re.findall(r'\b\w+\b|[^\w\s]', code)
        return tokens[:500]
    
    def build_vocab(self, codes):
        all_tokens = []
        for code in codes:
            tokens = self.tokenize_code(code)
            all_tokens.extend(tokens)
        
        unique_tokens = list(set(all_tokens))
        for token in unique_tokens[:self.vocab_size]:
            if token not in self.token_to_idx:
                self.token_to_idx[token] = self.current_idx
                self.idx_to_token[self.current_idx] = token
                self.current_idx += 1
    
    def build_ast_edges(self, tokens, seq_len):
        edges = np.zeros((seq_len, seq_len))
        for i in range(len(tokens) - 1):
            if i + 1 < seq_len:
                edges[i, i + 1] = 1
                edges[i + 1, i] = 1
        return edges
    
    def build_cfg_edges(self, tokens, seq_len):
        edges = np.zeros((seq_len, seq_len))
        control_keywords = {'if', 'else', 'for', 'while', 'switch', 'case', 'return'}
        for i, token in enumerate(tokens):
            if token in control_keywords and i < seq_len:
                for j in range(i + 1, min(i + 5, seq_len)):
                    edges[i, j] = 1
        return edges
    
    def build_dfg_edges(self, tokens, seq_len):
        dfg_r = np.zeros((seq_len, seq_len))
        dfg_w = np.zeros((seq_len, seq_len))
        dfg_c = np.zeros((seq_len, seq_len))
        
        var_positions = {}
        for i, token in enumerate(tokens):
            if i >= seq_len:
                break
            if token.isidentifier() and len(token) > 1:
                if token in var_positions:
                    for prev_pos in var_positions[token]:
                        dfg_r[i, prev_pos] = 1
                        dfg_w[prev_pos, i] = 1
                    var_positions[token].append(i)
                else:
                    var_positions[token] = [i]
                
                if i > 0 and tokens[i-1] == '=':
                    for j in range(max(0, i-5), i):
                        if tokens[j].isidentifier():
                            dfg_c[i, j] = 1
        
        return dfg_r, dfg_w, dfg_c
    
    def build_ncs_edges(self, tokens, seq_len):
        edges = np.zeros((seq_len, seq_len))
        for i in range(min(len(tokens) - 1, seq_len - 1)):
            edges[i, i + 1] = 1
        return edges
    
    def build_graph(self, code, max_len=128):
        tokens = self.tokenize_code(code)
        seq_len = min(len(tokens), max_len)
        
        token_ids = []
        for token in tokens[:max_len]:
            token_ids.append(self.token_to_idx.get(token, 0))
        
        while len(token_ids) < max_len:
            token_ids.append(0)
        
        ast_edges = self.build_ast_edges(tokens, seq_len)
        cfg_edges = self.build_cfg_edges(tokens, seq_len)
        dfg_r, dfg_w, dfg_c = self.build_dfg_edges(tokens, seq_len)
        ncs_edges = self.build_ncs_edges(tokens, seq_len)
        
        adj_matrices = np.stack([
            ast_edges,
            cfg_edges,
            dfg_r,
            dfg_w,
            dfg_c,
            ncs_edges
        ], axis=0)
        
        adj_padded = np.zeros((6, max_len, max_len))
        adj_padded[:, :seq_len, :seq_len] = adj_matrices[:, :seq_len, :seq_len]
        
        return np.array(token_ids), adj_padded

class Word2VecEmbedding:
    def __init__(self, embedding_dim=100):
        self.embedding_dim = embedding_dim
        self.model = None
        
    def train(self, codes):
        sentences = []
        for code in codes:
            tokens = re.findall(r'\b\w+\b|[^\w\s]', str(code).lower())
            if tokens:
                sentences.append(tokens[:500])
        
        self.model = Word2Vec(sentences, vector_size=self.embedding_dim, 
                            window=5, min_count=1, workers=4, sg=1)
    
    def get_embedding_matrix(self, vocab_size, token_to_idx):
        embedding_matrix = np.random.randn(vocab_size, self.embedding_dim).astype(np.float32) * 0.01
        
        for token, idx in token_to_idx.items():
            if idx < vocab_size and token in self.model.wv:
                embedding_matrix[idx] = self.model.wv[token]
        
        return embedding_matrix

class GatedGraphConv(nn.Module):
    def __init__(self, hidden_size, num_edge_types):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_edge_types = num_edge_types
        
        self.weight_ih = nn.ModuleList([
            nn.Linear(hidden_size, hidden_size) for _ in range(num_edge_types)
        ])
        
        self.gru = nn.GRUCell(hidden_size, hidden_size)
    
    def forward(self, x, adj_list):
        batch_size, num_nodes, _ = x.shape
        h = x
        
        for edge_type in range(self.num_edge_types):
            adj = adj_list[:, edge_type, :, :]
            
            a = torch.relu(self.weight_ih[edge_type](h))
            a = torch.bmm(adj, a)
            
            if edge_type == 0:
                aggregated = a
            else:
                aggregated = aggregated + a
        
        h_new = h.view(-1, self.hidden_size)
        aggregated = aggregated.view(-1, self.hidden_size)
        h_out = self.gru(aggregated, h_new)
        h_out = h_out.view(batch_size, num_nodes, self.hidden_size)
        
        return h_out

class ConvModule(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(kernel_size=3, stride=2)
        
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        
    def forward(self, h_with_init, h_only):
        h_with_init = h_with_init.transpose(1, 2)
        h_only = h_only.transpose(1, 2)
        
        z1 = self.conv1(h_with_init)
        z1 = torch.relu(z1)
        if z1.size(2) >= 3:
            z1 = self.pool1(z1)
        
        if z1.size(2) >= 1:
            z2 = self.conv2(z1)
            z2 = torch.relu(z2)
            if z2.size(2) >= 2:
                z2 = self.pool2(z2)
            z_final = z2
        else:
            z_final = z1
        
        y1 = self.conv1(h_only)
        y1 = torch.relu(y1)
        if y1.size(2) >= 3:
            y1 = self.pool1(y1)
        
        if y1.size(2) >= 1:
            y2 = self.conv2(y1)
            y2 = torch.relu(y2)
            if y2.size(2) >= 2:
                y2 = self.pool2(y2)
            y_final = y2
        else:
            y_final = y1
        
        z_final = torch.mean(z_final, dim=2)
        y_final = torch.mean(y_final, dim=2)
        
        output = z_final * y_final
        
        return output

class DevignModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, 
                 num_gnn_layers=6, num_edge_types=6, dropout=0.1, pretrained_embeddings=None):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained_embeddings))
        
        self.fc_init = nn.Linear(embedding_dim, hidden_size)
        
        self.gnn_layers = nn.ModuleList([
            GatedGraphConv(hidden_size, num_edge_types) for _ in range(num_gnn_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        
        self.conv_module = ConvModule(hidden_size)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )
    
    def forward(self, input_ids, adj_matrices, labels=None):
        x = self.embedding(input_ids)
        x_init = x
        
        h = self.fc_init(x)
        h = torch.relu(h)
        
        for gnn_layer in self.gnn_layers:
            h = gnn_layer(h, adj_matrices)
        
        h = self.dropout(h)
        
        h_with_init = torch.cat([h, x_init], dim=-1)
        h_with_init = h_with_init[:, :, :h.size(-1)]
        
        graph_embedding = self.conv_module(h_with_init, h)
        
        logits = self.classifier(graph_embedding)
        
        outputs = {'logits': logits}
        
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            outputs['loss'] = loss
        
        return outputs

class CodeGraphDataset(Dataset):
    def __init__(self, df, graph_builder, max_len=128):
        self.df = df.reset_index(drop=True)
        self.graph_builder = graph_builder
        self.max_len = max_len
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        code = str(self.df.iloc[idx]['func'])
        label = int(self.df.iloc[idx]['label'])
        
        token_ids, adj_matrices = self.graph_builder.build_graph(code, self.max_len)
        
        return {
            'input_ids': torch.tensor(token_ids, dtype=torch.long),
            'adj_matrices': torch.tensor(adj_matrices, dtype=torch.float32),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def evaluate_model(model, dataloader, device, num_classes):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            adj_matrices = batch['adj_matrices'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, adj_matrices)
            logits = outputs['logits']
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    
    cm = confusion_matrix(all_labels, all_preds)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    
    if num_classes == 2:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    else:
        try:
            y_bin = label_binarize(all_labels, classes=range(num_classes))
            auc = roc_auc_score(y_bin, all_probs, average='weighted', multi_class='ovr')
        except:
            auc = 0.0
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': int(tp),
        'FN': int(fn)
    }

def main():
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    num_classes = len(train_df['label'].unique())
    print(f"Number of classes: {num_classes}")
    print(f"Training samples: {len(train_df)}")
    print(f"Testing samples: {len(test_df)}\n")
    
    print("Building vocabulary and graph structures...")
    graph_builder = CompositeGraphBuilder(vocab_size=10000)
    graph_builder.build_vocab(train_df['func'].tolist())
    
    print("Training Word2Vec embeddings...")
    w2v = Word2VecEmbedding(embedding_dim=100)
    w2v.train(train_df['func'].tolist())
    embedding_matrix = w2v.get_embedding_matrix(len(graph_builder.token_to_idx), graph_builder.token_to_idx)
    
    print("Creating datasets...")
    train_dataset = CodeGraphDataset(train_df, graph_builder, max_len=128)
    test_dataset = CodeGraphDataset(test_df, graph_builder, max_len=128)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    print("Initializing Devign model...")
    model = DevignModel(
        vocab_size=len(graph_builder.token_to_idx),
        embedding_dim=100,
        hidden_size=200,
        num_classes=num_classes,
        num_gnn_layers=6,
        num_edge_types=6,
        dropout=0.1,
        pretrained_embeddings=embedding_matrix
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
    
    print("\nStarting training...\n")
    start_time = time.time()
    
    num_epochs = 5
    best_f1 = 0
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        epoch_start = time.time()
        
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            adj_matrices = batch['adj_matrices'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, adj_matrices, labels)
            loss = outputs['loss']
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Time: {epoch_time:.2f}s")
        
        if (epoch + 1) % 2 == 0:
            metrics = evaluate_model(model, test_loader, device, num_classes)
            print(f"  Validation F1: {metrics['F1']:.4f}, Accuracy: {metrics['Accuracy']:.4f}")
            
            if metrics['F1'] > best_f1:
                best_f1 = metrics['F1']
                torch.save(model.state_dict(), 'best_devign_model.pth')
    
    total_training_time = time.time() - start_time
    
    print("\n" + "="*70)
    print("TRAINING COMPLETED")
    print("="*70)
    print(f"Total Training Time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)\n")
    
    print("Loading best model for final evaluation...")
    model.load_state_dict(torch.load('best_devign_model.pth'))
    
    print("\n" + "="*70)
    print("FINAL TEST RESULTS")
    print("="*70)
    
    eval_start = time.time()
    final_metrics = evaluate_model(model, test_loader, device, num_classes)
    eval_time = time.time() - eval_start
    
    for metric, value in final_metrics.items():
        if isinstance(value, float):
            print(f"{metric}: {value:.4f}")
        else:
            print(f"{metric}: {value}")
    
    print(f"\nEvaluation Time: {eval_time:.2f} seconds")
    print(f"Total Execution Time: {(time.time() - start_time):.2f} seconds")
    print("="*70)

if __name__ == "__main__":
    main()

# IVDetect

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    matthews_corrcoef, cohen_kappa_score, confusion_matrix,
    roc_auc_score, mean_squared_error, mean_absolute_error
)
from sklearn.preprocessing import label_binarize
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
import time
import warnings
warnings.filterwarnings('ignore')

class ContextAwareCodeEmbedding:
    def __init__(self, vector_size=64):
        self.vector_size = vector_size
        self.w2v_model = None
        
    def fit_word2vec(self, code_snippets):
        tokenized = [word_tokenize(str(code).lower()) for code in code_snippets]
        self.w2v_model = Word2Vec(tokenized, vector_size=self.vector_size, 
                                   window=5, min_count=1, workers=4, epochs=10)
        
    def extract_subtokens(self, code):
        tokens = word_tokenize(str(code).lower())
        subtokens = []
        for token in tokens:
            if any(c.isupper() for c in token):
                parts = []
                current = []
                for c in token:
                    if c.isupper() and current:
                        parts.append(''.join(current))
                        current = [c.lower()]
                    else:
                        current.append(c.lower())
                if current:
                    parts.append(''.join(current))
                subtokens.extend(parts)
            else:
                subtokens.append(token)
        return subtokens
    
    def get_token_embedding(self, code):
        tokens = self.extract_subtokens(code)
        embeddings = []
        for token in tokens:
            if token in self.w2v_model.wv:
                embeddings.append(self.w2v_model.wv[token])
        if len(embeddings) == 0:
            return np.zeros(self.vector_size, dtype=np.float32)
        return np.mean(embeddings, axis=0).astype(np.float32)
    
    def extract_variables(self, code):
        tokens = word_tokenize(str(code))
        variables = [t for t in tokens if t.isidentifier() and not t.isupper()]
        var_embeddings = []
        for var in variables[:10]:
            subtokens = self.extract_subtokens(var)
            for st in subtokens:
                if st in self.w2v_model.wv:
                    var_embeddings.append(self.w2v_model.wv[st])
        if len(var_embeddings) == 0:
            return np.zeros(self.vector_size, dtype=np.float32)
        return np.mean(var_embeddings, axis=0).astype(np.float32)
    
    def simulate_data_dependency_context(self, code):
        tokens = self.extract_subtokens(code)
        context_tokens = tokens[:len(tokens)//2] if len(tokens) > 1 else tokens
        embeddings = []
        for token in context_tokens:
            if token in self.w2v_model.wv:
                embeddings.append(self.w2v_model.wv[token])
        if len(embeddings) == 0:
            return np.zeros(self.vector_size, dtype=np.float32)
        return np.mean(embeddings, axis=0).astype(np.float32)
    
    def simulate_control_dependency_context(self, code):
        keywords = ['if', 'else', 'while', 'for', 'return', 'break', 'continue']
        tokens = self.extract_subtokens(code)
        control_tokens = [t for t in tokens if t in keywords]
        if not control_tokens:
            control_tokens = tokens[-len(tokens)//2:] if len(tokens) > 1 else tokens
        embeddings = []
        for token in control_tokens:
            if token in self.w2v_model.wv:
                embeddings.append(self.w2v_model.wv[token])
        if len(embeddings) == 0:
            return np.zeros(self.vector_size, dtype=np.float32)
        return np.mean(embeddings, axis=0).astype(np.float32)
    
    def get_context_aware_embedding(self, code):
        f1_token = self.get_token_embedding(code)
        f2_ast = self.get_token_embedding(code) * 0.8
        f3_var = self.extract_variables(code)
        f4_data_dep = self.simulate_data_dependency_context(code)
        f5_ctrl_dep = self.simulate_control_dependency_context(code)
        
        features = np.concatenate([f1_token, f2_ast, f3_var, f4_data_dep, f5_ctrl_dep])
        return features.astype(np.float32)

def generate_pdg_edges(num_nodes):
    if num_nodes == 1:
        return torch.tensor([[0], [0]], dtype=torch.long)
    
    edge_index = []
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                distance = abs(i - j)
                if distance <= 2 or np.random.random() < 0.3:
                    edge_index.append([i, j])
    
    if len(edge_index) == 0:
        edge_index = [[i, (i+1) % num_nodes] for i in range(num_nodes)]
        edge_index += [[(i+1) % num_nodes, i] for i in range(num_nodes)]
    
    return torch.tensor(edge_index, dtype=torch.long).t().contiguous()

def prepare_graph_data(embeddings, labels):
    data_list = []
    for i in range(len(embeddings)):
        try:
            x = torch.tensor(embeddings[i], dtype=torch.float32).unsqueeze(0)
            edge_index = generate_pdg_edges(x.shape[0])
            y = torch.tensor([int(labels.iloc[i])], dtype=torch.long)
            data = Data(x=x, edge_index=edge_index, y=y)
            data_list.append(data)
        except Exception as e:
            continue
    return data_list

class AttentionBiGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(AttentionBiGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        gru_out, _ = self.gru(x)
        attn_weights = torch.softmax(self.attention(gru_out), dim=1)
        context = torch.sum(attn_weights * gru_out, dim=1)
        return context

class FeatureAttentionGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FeatureAttentionGCN, self).__init__()
        self.attention_gru = AttentionBiGRU(input_dim, hidden_dim // 4)
        
        self.conv1 = GATConv(input_dim, hidden_dim, heads=4, concat=True, dropout=0.3)
        self.conv2 = GATConv(hidden_dim * 4, hidden_dim, heads=1, concat=False, dropout=0.3)
        
        self.dropout = nn.Dropout(0.4)
        self.bn1 = nn.LayerNorm(hidden_dim * 4)
        self.bn2 = nn.LayerNorm(hidden_dim)
        
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, output_dim)


        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout(x)
        
        x = global_mean_pool(x, batch)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_multiclass(model, loader, device, num_classes):
    model.eval()
    y_true, y_pred, y_scores = [], [], []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            probs = torch.softmax(out, dim=1)
            pred = out.argmax(dim=1)
            
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            y_scores.extend(probs.cpu().numpy())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_scores = np.array(y_scores)
    
    cm = confusion_matrix(y_true, y_pred)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    
    y_true_bin = label_binarize(y_true, classes=range(num_classes))
    try:
        auc = roc_auc_score(y_true_bin, y_scores, average='weighted', multi_class='ovr')
    except:
        auc = 0.0
    
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    tp = np.sum(np.diag(cm))
    fn = np.sum(y_true != y_pred)
    
    return {
        'confusion_matrix': cm,
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn
    }

def main():
    start_time = time.time()
    
    device = torch.device("cpu" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    train_path = '/Users/akter/fahim/data/trainpro (1).csv'
    test_path = '/Users/akter/fahim/data/testpro.csv'
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
 
    
    X_train = train_df['func'].fillna('')
    y_train = train_df['label'].astype(int)
    X_test = test_df['func'].fillna('')
    y_test = test_df['label'].astype(int)
    
    num_classes = len(y_train.unique())
    print(f"Number of classes: {num_classes}")
    print(f"Training samples: {len(train_df)}")
    print(f"Testing samples: {len(test_df)}\n")
    
    print("Building context-aware embeddings...")
    embedder = ContextAwareCodeEmbedding(vector_size=64)
    embedder.fit_word2vec(X_train)
    
    train_embeddings = [embedder.get_context_aware_embedding(code) for code in X_train]
    test_embeddings = [embedder.get_context_aware_embedding(code) for code in X_test]
    
    print("Preparing graph data...")
    train_data = prepare_graph_data(train_embeddings, y_train)
    test_data = prepare_graph_data(test_embeddings, y_test)
    
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
    
    input_dim = train_data[0].x.shape[1]
    print(f"Input dimension: {input_dim}\n")
    
    model = FeatureAttentionGCN(input_dim=input_dim, hidden_dim=128, output_dim=num_classes)
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    print("Training FA-GCN model...")
    best_loss = float('inf')
    patience_counter = 0
    max_patience = 15
    
    for epoch in range(1, 6):
        loss = train_epoch(model, train_loader, optimizer, criterion, device)
        scheduler.step(loss)
        
        if loss < best_loss:
            best_loss = loss
            torch.save(model.state_dict(), 'ivdetect_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= max_patience:
            print(f"Early stopping at epoch {epoch}")
            break
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Best Loss: {best_loss:.4f}")
    
    print("\nLoading best model and evaluating...")
    model.load_state_dict(torch.load('ivdetect_model.pth'))
    
    test_metrics = evaluate_multiclass(model, test_loader, device, num_classes)
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    print("\n" + "="*60)
    print("FINAL TEST RESULTS")
    print("="*60)
    print(f"Execution Time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")
    print(f"\nAUC:       {test_metrics['AUC']:.4f}")
    print(f"Accuracy:  {test_metrics['Accuracy']:.4f}")
    print(f"Precision: {test_metrics['Precision']:.4f}")
    print(f"Recall:    {test_metrics['Recall']:.4f}")
    print(f"F1:        {test_metrics['F1']:.4f}")
    print(f"MCC:       {test_metrics['MCC']:.4f}")
    print(f"Kappa:     {test_metrics['Kappa']:.4f}")
    print(f"MSE:       {test_metrics['MSE']:.4f}")
    print(f"MAE:       {test_metrics['MAE']:.4f}")
    print(f"TP:        {test_metrics['TP']}")
    print(f"FN:        {test_metrics['FN']}")
    print(f"\nConfusion Matrix:\n{test_metrics['confusion_matrix']}")
    print("="*60)

if __name__ == "__main__":
    main()

Device: cpu

Loading datasets...
Number of classes: 6
Training samples: 21793
Testing samples: 5400

Building context-aware embeddings...
Preparing graph data...
Input dimension: 320

Training FA-GCN model...

Loading best model and evaluating...

FINAL TEST RESULTS
Execution Time: 107.25 seconds (1.79 minutes)

AUC:       0.8395
Accuracy:  0.5061
Precision: 0.5786
Recall:    0.5061
F1:        0.4998
MCC:       0.4229
Kappa:     0.4087
MSE:       3.5678
MAE:       1.2056
TP:        2733
FN:        2667

Confusion Matrix:
[[818  18  26   1   2  35]
 [ 52 223 237   5 102 315]
 [ 36  17 481   3  21 302]
 [ 94  34  69 407  21 293]
 [ 33 177 167   6 243 283]
 [ 38  59 194   2  25 561]]


# vulexplainer paper with draper 6


In [ ]:

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                              confusion_matrix, roc_auc_score, matthews_corrcoef, cohen_kappa_score,
                              mean_squared_error, mean_absolute_error)
from tree_sitter import Language, Parser
import tree_sitter_python
import re
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f'Device: {device}\n')

train_path = '/Users/akter/fahim/data/trainpro (1).csv'
test_path = '/Users/akter/fahim/data/testpro.csv'

print("Loading datasets...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)


print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Label distribution in train:\n{train_df['label'].value_counts().sort_index()}\n")
print(f"Label distribution in test:\n{test_df['label'].value_counts().sort_index()}\n")

NUM_CLASSES = 6
label_freq = train_df['label'].value_counts().to_dict()
print(f"Label frequencies: {label_freq}\n")

def compute_abstract_types(label_freq, num_classes):
    sorted_labels = sorted(label_freq.items(), key=lambda x: x[1], reverse=True)
    label_to_group = {}
    num_groups = 3
    labels_per_group = num_classes // num_groups
    for idx, (label, freq) in enumerate(sorted_labels):
        group_id = min(idx // labels_per_group, num_groups - 1)
        label_to_group[label] = group_id
    group_to_labels = {}
    for label, group in label_to_group.items():
        if group not in group_to_labels:
            group_to_labels[group] = []
        group_to_labels[group].append(label)
    for group in group_to_labels:
        group_to_labels[group] = sorted(group_to_labels[group])
    return label_to_group, group_to_labels

label_to_group, group_to_labels = compute_abstract_types(label_freq, NUM_CLASSES)
print(f"Hierarchical Grouping (Abstract Types):")
for group, labels in group_to_labels.items():
    print(f"  Group {group}: {labels}")
print()

class DFGExtractor:
    def __init__(self):
        try:
            self.parser = Parser()
            PY_LANGUAGE = Language(tree_sitter_python.language())
            self.parser.set_language(PY_LANGUAGE)
            self.available = True
        except:
            self.parser = None
            self.available = False
            print("Warning: TreeSitter not available, using fallback DFG extraction")
    
    def extract_dataflow(self, code, max_nodes=128):
        if not self.available:
            return self._fallback_dfg(code, max_nodes)
        try:
            tree = self.parser.parse(bytes(code, "utf8"))
            root_node = tree.root_node
            dfg_nodes = []
            dfg_edges = []
            variable_def_map = {}
            variable_use_map = {}
            
            def get_line_number(node):
                return node.start_point[0]
            
            def traverse_def_use(node, parent_scope="global"):
                if len(dfg_nodes) >= max_nodes:
                    return
                node_type = node.type
                if node_type in ['identifier', 'attribute']:
                    var_name = code[node.start_byte:node.end_byte]
                    line_num = get_line_number(node)
                    parent_type = node.parent.type if node.parent else None
                    if parent_type in ['assignment', 'augmented_assignment', 'function_definition', 'parameters']:
                        op_type = 'DEF'
                        node_id = len(dfg_nodes)
                        dfg_nodes.append({
                            'id': node_id,
                            'name': var_name,
                            'type': op_type,
                            'line': line_num,
                            'scope': parent_scope
                        })
                        if var_name not in variable_def_map:
                            variable_def_map[var_name] = []
                        variable_def_map[var_name].append(node_id)
                    else:
                        op_type = 'USE'
                        node_id = len(dfg_nodes)
                        dfg_nodes.append({
                            'id': node_id,
                            'name': var_name,
                            'type': op_type,
                            'line': line_num,
                            'scope': parent_scope
                        })
                        if var_name not in variable_use_map:
                            variable_use_map[var_name] = []
                        variable_use_map[var_name].append(node_id)
                        if var_name in variable_def_map:
                            for def_id in variable_def_map[var_name]:
                                dfg_edges.append((def_id, node_id, 'def-use'))
                elif node_type in ['if_statement', 'while_statement', 'for_statement']:
                    condition_node = node.child_by_field_name('condition')
                    if condition_node:
                        traverse_def_use(condition_node, parent_scope)
                    body_node = node.child_by_field_name('body')
                    if body_node:
                        for child in body_node.children:
                            traverse_def_use(child, parent_scope + f"_{node_type}")
                    return
                for child in node.children:
                    traverse_def_use(child, parent_scope)
            
            traverse_def_use(root_node)
            for var_name in variable_use_map:
                uses = variable_use_map[var_name]
                for i in range(len(uses) - 1):
                    dfg_edges.append((uses[i], uses[i+1], 'use-use'))
            return dfg_nodes, dfg_edges
        except Exception as e:
            return self._fallback_dfg(code, max_nodes)
    
    def _fallback_dfg(self, code, max_nodes):
        lines = code.split('\n')[:max_nodes]
        dfg_nodes = []
        dfg_edges = []
        var_pattern = re.compile(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b')
        var_def_map = {}
        for line_idx, line in enumerate(lines):
            if '=' in line and not line.strip().startswith('#'):
                parts = line.split('=')
                if len(parts) >= 2:
                    left = parts[0].strip()
                    right = '='.join(parts[1:]).strip()
                    def_vars = var_pattern.findall(left)
                    for var in def_vars:
                        node_id = len(dfg_nodes)
                        dfg_nodes.append({
                            'id': node_id,
                            'name': var,
                            'type': 'DEF',
                            'line': line_idx,
                            'scope': 'global'
                        })
                        var_def_map[var] = node_id
                    use_vars = var_pattern.findall(right)
                    for var in use_vars:
                        node_id = len(dfg_nodes)
                        dfg_nodes.append({
                            'id': node_id,
                            'name': var,
                            'type': 'USE',
                            'line': line_idx,
                            'scope': 'global'
                        })
                        if var in var_def_map:
                            dfg_edges.append((var_def_map[var], node_id, 'def-use'))
            else:
                vars_in_line = var_pattern.findall(line)
                for var in vars_in_line:
                    node_id = len(dfg_nodes)
                    dfg_nodes.append({
                        'id': node_id,
                        'name': var,
                        'type': 'USE',
                        'line': line_idx,
                        'scope': 'global'
                    })
                    if var in var_def_map:
                        dfg_edges.append((var_def_map[var], node_id, 'def-use'))
        return dfg_nodes, dfg_edges

class GraphCodeBERTTokenizer:
    def __init__(self):
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.dis_token = '<DIS>'
        self.tokenizer.add_tokens([self.dis_token])
        self.dis_token_id = self.tokenizer.convert_tokens_to_ids(self.dis_token)
        self.dfg_extractor = DFGExtractor()
    
    def encode_with_dfg(self, code, max_length=512):
        dfg_nodes, dfg_edges = self.dfg_extractor.extract_dataflow(code, max_nodes=64)
        tokens = self.tokenizer.tokenize(code)[:max_length-3]
        tokens = [self.tokenizer.cls_token] + tokens + [self.dis_token] + [self.tokenizer.sep_token]
        input_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        seq_len = len(input_ids)
        attention_mask = [1] * seq_len
        code_tokens_len = seq_len - 3
        dfg_node_positions = {}
        for idx, node in enumerate(dfg_nodes):
            if idx < code_tokens_len:
                dfg_node_positions[node['id']] = idx + 1
        dfg_to_code = {i: [] for i in range(seq_len)}
        code_to_dfg = {i: [] for i in range(len(dfg_nodes))}
        dfg_to_dfg = {i: [] for i in range(len(dfg_nodes))}
        for src, tgt, edge_type in dfg_edges:
            if src in dfg_node_positions and tgt in dfg_node_positions:
                src_pos = dfg_node_positions[src]
                tgt_pos = dfg_node_positions[tgt]
                dfg_to_code[src_pos].append(tgt_pos)
                code_to_dfg[tgt].append(src_pos)
                dfg_to_dfg[src].append(tgt)
        padding_length = max_length - seq_len
        input_ids += [self.tokenizer.pad_token_id] * padding_length
        attention_mask += [0] * padding_length
        attn_mask = self._create_graph_guided_mask(seq_len, dfg_to_code, code_to_dfg, dfg_to_dfg, max_length)
        return {
            'input_ids': torch.tensor(input_ids[:max_length]),
            'attention_mask': torch.tensor(attention_mask[:max_length]),
            'attn_mask': attn_mask,
            'seq_len': seq_len
        }
    
    def _create_graph_guided_mask(self, seq_len, dfg_to_code, code_to_dfg, dfg_to_dfg, max_length):
        attn_mask = torch.zeros((max_length, max_length), dtype=torch.float)
        for i in range(min(seq_len, max_length)):
            for j in range(min(seq_len, max_length)):
                attn_mask[i, j] = 1.0
        for src_pos in dfg_to_code:
            if src_pos < max_length:
                for tgt_pos in dfg_to_code[src_pos]:
                    if tgt_pos < max_length:
                        attn_mask[src_pos, tgt_pos] = 1.0
        for tgt_node in code_to_dfg:
            for src_pos in code_to_dfg[tgt_node]:
                if src_pos < max_length and tgt_node < max_length:
                    attn_mask[src_pos, tgt_node] = 1.0
        for src_node in dfg_to_dfg:
            for tgt_node in dfg_to_dfg[src_node]:
                if src_node < max_length and tgt_node < max_length:
                    attn_mask[src_node, tgt_node] = 1.0
        return attn_mask

class VulnerabilityDataset(Dataset):
    def __init__(self, dataframe, tokenizer_wrapper, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer_wrapper
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        code = str(self.data.loc[idx, 'func'])
        label = int(self.data.loc[idx, 'label'])
        encoding = self.tokenizer.encode_with_dfg(code, self.max_length)
        encoding['label'] = torch.tensor(label, dtype=torch.long)
        return encoding

class TextCNNTeacher(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_filters=100, filter_sizes=[3, 4, 5], num_classes_per_group=None, dropout=0.1):
        super(TextCNNTeacher, self).__init__()
        if num_classes_per_group is None:
            num_classes_per_group = [2, 2, 2]
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=1)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        total_filters = num_filters * len(filter_sizes)
        self.classifiers = nn.ModuleList([
            nn.Linear(total_filters, num_classes) for num_classes in num_classes_per_group
        ])
        self.num_groups = len(num_classes_per_group)
        self.num_classes_per_group = num_classes_per_group
    
    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)
        conv_outputs = []
        for conv in self.convs:
            conv_out = F.relu(conv(x))
            pooled = F.max_pool1d(conv_out, conv_out.size(2)).squeeze(2)
            conv_outputs.append(pooled)
        features = torch.cat(conv_outputs, dim=1)
        features = self.dropout(features)
        logits_list = [classifier(features) for classifier in self.classifiers]
        return logits_list, features

class GraphCodeBERTStudent(nn.Module):
    def __init__(self, num_classes=6, dropout=0.1, tokenizer_size=None):
        super(GraphCodeBERTStudent, self).__init__()
        self.config = RobertaConfig.from_pretrained('microsoft/graphcodebert-base')
        self.encoder = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        if tokenizer_size:
            self.encoder.resize_token_embeddings(tokenizer_size)
        hidden_size = self.config.hidden_size
        self.cls_classifier = nn.Linear(hidden_size, num_classes)
        self.dis_classifier = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.num_classes = num_classes
        self.dis_token_id = tokenizer_size - 1 if tokenizer_size else 50265
    
    def forward(self, input_ids, attention_mask, attn_mask=None):
        batch_size = input_ids.size(0)
        if attn_mask is not None and attn_mask.dim() == 2:
            attn_mask = attn_mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, -1, -1)
        outputs = self.encoder(input_ids=input_ids, attention_mask=attn_mask if attn_mask is not None else attention_mask)
        hidden_states = outputs.last_hidden_state
        cls_output = hidden_states[:, 0, :]
        dis_positions = (input_ids == self.dis_token_id).nonzero(as_tuple=True)
        if len(dis_positions[0]) > 0:
            batch_idx = dis_positions[0]
            token_idx = dis_positions[1]
            dis_output = torch.zeros(batch_size, hidden_states.size(2), device=hidden_states.device)
            for b in range(batch_size):
                mask = batch_idx == b
                if mask.sum() > 0:
                    pos = token_idx[mask][0]
                    dis_output[b] = hidden_states[b, pos, :]
                else:
                    seq_len = attention_mask[b].sum() - 1
                    dis_output[b] = hidden_states[b, seq_len, :]
        else:
            seq_lengths = attention_mask.sum(dim=1) - 1
            dis_output = hidden_states[torch.arange(batch_size), seq_lengths, :]
        cls_output = self.dropout(cls_output)
        dis_output = self.dropout(dis_output)
        cls_logits = self.cls_classifier(cls_output)
        dis_logits = self.dis_classifier(dis_output)
        return cls_logits, dis_logits

def train_teacher_epoch(model, dataloader, optimizer, device, group_to_labels, label_to_group):
    model.train()
    total_loss = 0
    group_losses = {g: 0 for g in group_to_labels.keys()}
    group_counts = {g: 0 for g in group_to_labels.keys()}
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        logits_list, _ = model(input_ids)
        batch_loss = 0
        num_active_groups = 0
        for group_id in range(model.num_groups):
            group_labels_list = group_to_labels[group_id]
            group_mask = torch.tensor([l.item() in group_labels_list for l in labels], dtype=torch.bool, device=device)
            if group_mask.sum() > 0:
                group_labels = labels[group_mask]
                label_to_idx = {label: idx for idx, label in enumerate(group_labels_list)}
                mapped_labels = torch.tensor([label_to_idx[l.item()] for l in group_labels], device=device)
                group_logits = logits_list[group_id][group_mask]
                group_loss = F.cross_entropy(group_logits, mapped_labels)
                batch_loss += group_loss
                num_active_groups += 1
                group_losses[group_id] += group_loss.item()
                group_counts[group_id] += 1
        if num_active_groups > 0:
            batch_loss = batch_loss / num_active_groups
            optimizer.zero_grad()
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += batch_loss.item()
    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    return avg_loss

def train_student_epoch(model, teacher_model, dataloader, optimizer, device, lambda_distill, temperature, group_to_labels, group_freq):
    model.train()
    teacher_model.eval()
    total_loss = 0
    total_loss_cls = 0
    total_loss_distill_per_group = {g: 0 for g in group_to_labels.keys()}
    total_samples_per_group = {g: 0 for g in group_to_labels.keys()}
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        attn_mask = batch['attn_mask'].to(device)
        labels = batch['label'].to(device)
        cls_logits, dis_logits = model(input_ids, attention_mask, attn_mask)
        loss_cls = F.cross_entropy(cls_logits, labels)
        with torch.no_grad():
            teacher_logits_list, _ = teacher_model(input_ids)
        total_distill_loss = 0
        for group_id, group_labels_list in group_to_labels.items():
            group_mask = torch.tensor([l.item() in group_labels_list for l in labels], dtype=torch.bool, device=device)
            if group_mask.sum() > 0:
                group_dis_logits = dis_logits[group_mask]
                teacher_group_logits = teacher_logits_list[group_id][group_mask]
                group_global_logits = torch.zeros(group_mask.sum(), model.num_classes, device=device)
                for local_idx, global_label in enumerate(group_labels_list):
                    group_global_logits[:, global_label] = teacher_group_logits[:, local_idx]
                weight = group_freq[group_id]
                kl_loss = F.kl_div(
                    F.log_softmax(group_dis_logits / temperature, dim=1),
                    F.softmax(group_global_logits / temperature, dim=1),
                    reduction='batchmean'
                ) * (temperature ** 2)
                total_distill_loss += weight * kl_loss
                total_loss_distill_per_group[group_id] += kl_loss.item()
                total_samples_per_group[group_id] += 1
        loss = (1 - lambda_distill) * loss_cls + lambda_distill * total_distill_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        total_loss_cls += loss_cls.item()
    avg_loss = total_loss / len(dataloader)
    avg_loss_cls = total_loss_cls / len(dataloader)
    return avg_loss, avg_loss_cls

def evaluate_student(model, dataloader, device, eta=0.9):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            attn_mask = batch['attn_mask'].to(device)
            labels = batch['label'].to(device)
            cls_logits, dis_logits = model(input_ids, attention_mask, attn_mask)
            cls_probs = F.softmax(cls_logits, dim=1)
            dis_probs = F.softmax(dis_logits, dim=1)
            combined_probs = eta * cls_probs + (1 - eta) * dis_probs
            preds = torch.argmax(combined_probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(combined_probs.cpu().numpy())
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0
    mcc = matthews_corrcoef(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    tp = np.diag(cm).sum()
    fn = cm.sum() - tp
    metrics = {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': int(tp),
        'FN': int(fn)
    }
    return metrics, all_preds, all_labels

def main():
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    LEARNING_RATE_TEACHER = 5e-3
    LEARNING_RATE_STUDENT = 2e-5
    EPOCHS_TEACHER = 5
    EPOCHS_STUDENT = 5
    LAMBDA_DISTILL = 0.7
    ETA = 0.9
    print("Initializing GraphCodeBERT tokenizer with DFG support...")
    tokenizer_wrapper = GraphCodeBERTTokenizer()
    vocab_size = len(tokenizer_wrapper.tokenizer)
    print(f"Vocabulary size: {vocab_size}")
    print("\nCreating datasets with DFG extraction...")
    train_dataset = VulnerabilityDataset(train_df, tokenizer_wrapper, MAX_LENGTH)
    test_dataset = VulnerabilityDataset(test_df, tokenizer_wrapper, MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    num_classes_per_group = [len(group_to_labels[g]) for g in sorted(group_to_labels.keys())]
    total_samples = sum(label_freq.values())
    group_freq = {}
    for group_id in group_to_labels:
        group_sample_count = sum(label_freq[label] for label in group_to_labels[group_id])
        group_freq[group_id] = group_sample_count / total_samples
    print(f"Classes per group: {num_classes_per_group}")
    print(f"Group frequency weights: {group_freq}")
    print("\n" + "="*70)
    print("PHASE 1: Training TextCNN Teacher with Multi-Head Design")
    print("="*70)
    teacher_model = TextCNNTeacher(
        vocab_size=vocab_size,
        embed_dim=768,
        num_filters=100,
        filter_sizes=[3, 4, 5],
        num_classes_per_group=num_classes_per_group,
        dropout=0.1
    ).to(device)
    print(f"Teacher parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")
    teacher_optimizer = torch.optim.AdamW(teacher_model.parameters(), lr=LEARNING_RATE_TEACHER)
    teacher_scheduler = torch.optim.lr_scheduler.LinearLR(teacher_optimizer, start_factor=1.0, end_factor=0.1, total_iters=EPOCHS_TEACHER)
    best_teacher_loss = float('inf')
    for epoch in range(EPOCHS_TEACHER):
        train_loss = train_teacher_epoch(teacher_model, train_loader, teacher_optimizer, device, group_to_labels, label_to_group)
        teacher_scheduler.step()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS_TEACHER} - Loss: {train_loss:.4f}")
        if train_loss < best_teacher_loss:
            best_teacher_loss = train_loss
            torch.save(teacher_model.state_dict(), 'best_teacher_model.pt')
    teacher_model.load_state_dict(torch.load('best_teacher_model.pt'))
    print(f"\nBest Teacher Loss: {best_teacher_loss:.4f}")
    print("\n" + "="*70)
    print("PHASE 2: Hierarchical Distillation Training")
    print("="*70)
    student_model = GraphCodeBERTStudent(
        num_classes=NUM_CLASSES,
        dropout=0.1,
        tokenizer_size=vocab_size
    ).to(device)
    print(f"Student parameters: {sum(p.numel() for p in student_model.parameters()):,}")
    student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=LEARNING_RATE_STUDENT)
    student_scheduler = torch.optim.lr_scheduler.LinearLR(student_optimizer, start_factor=1.0, end_factor=0.1, total_iters=EPOCHS_STUDENT)
    best_f1 = 0
    best_metrics = None
    print("\nTraining Student Model with Hierarchical Distillation...")
    for epoch in range(EPOCHS_STUDENT):
        train_loss, loss_cls = train_student_epoch(
            student_model, teacher_model, train_loader, student_optimizer,
            device, LAMBDA_DISTILL, temperature=1.0, group_to_labels=group_to_labels, group_freq=group_freq
        )
        student_scheduler.step()
        metrics, preds, labels = evaluate_student(student_model, test_loader, device, ETA)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS_STUDENT}")
            print(f"  Loss: {train_loss:.4f} (CE: {loss_cls:.4f})")
            print(f"  Acc: {metrics['Accuracy']:.4f} | F1: {metrics['F1']:.4f} | AUC: {metrics['AUC']:.4f}")
        if metrics['F1'] > best_f1:
            best_f1 = metrics['F1']
            best_metrics = metrics
            torch.save(student_model.state_dict(), 'best_student_model.pt')
    print("\n" + "="*70)
    print("FINAL EVALUATION - VulExplainer Results")
    print("="*70)
    student_model.load_state_dict(torch.load('best_student_model.pt'))
    metrics, predictions, labels = evaluate_student(student_model, test_loader, device, ETA)
    print(f"\nOverall Test Metrics:")
    print(f"  AUC:       {metrics['AUC']:.4f}")
    print(f"  Accuracy:  {metrics['Accuracy']:.4f}")
    print(f"  Precision: {metrics['Precision']:.4f}")
    print(f"  Recall:    {metrics['Recall']:.4f}")
    print(f"  F1:        {metrics['F1']:.4f}")
    print(f"  MCC:       {metrics['MCC']:.4f}")
    print(f"  Kappa:     {metrics['Kappa']:.4f}")
    print(f"  MSE:       {metrics['MSE']:.4f}")
    print(f"  MAE:       {metrics['MAE']:.4f}")
    print(f"  TP:        {metrics['TP']}")
    print(f"  FN:        {metrics['FN']}")
    print(f"\nPer-Class Metrics:")
    for class_id in range(NUM_CLASSES):
        class_mask = np.array(labels) == class_id
        if class_mask.sum() > 0:
            class_preds = np.array(predictions)[class_mask]
            class_labels = np.array(labels)[class_mask]
            class_acc = accuracy_score(class_labels, class_preds)
            class_prec, class_rec, class_f1, _ = precision_recall_fscore_support(
                class_labels, class_preds, average='binary', pos_label=class_id, zero_division=0
            )
            print(f"  Class {class_id}: Acc={class_acc:.4f} | Prec={class_prec:.4f} | Rec={class_rec:.4f} | F1={class_f1:.4f} ({class_mask.sum()} samples)")
    group_metrics = {}
    for group_id in group_to_labels:
        group_labels_list = group_to_labels[group_id]
        group_mask = [l in group_labels_list for l in labels]
        group_labels_filtered = [l for l, m in zip(labels, group_mask) if m]
        group_preds_filtered = [p for p, m in zip(predictions, group_mask) if m]
        if len(group_labels_filtered) > 0:
            acc = accuracy_score(group_labels_filtered, group_preds_filtered)
            prec, rec, f1_g, _ = precision_recall_fscore_support(
                group_labels_filtered, group_preds_filtered, average='weighted', zero_division=0
            )
            group_metrics[f'Group_{group_id}'] = {
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1': f1_g,
                'Samples': len(group_labels_filtered)
            }
    print(f"\nPer-Group Metrics (Abstract Types):")
    for group_name, group_met in sorted(group_metrics.items()):
        print(f"  {group_name}: Acc={group_met['Accuracy']:.4f} | Prec={group_met['Precision']:.4f} | Rec={group_met['Recall']:.4f} | F1={group_met['F1']:.4f} ({group_met['Samples']} samples)")
    print(f"\nDetailed Classification Report:")
    print(classification_report(labels, predictions, digits=4, zero_division=0))
    cm = confusion_matrix(labels, predictions)
    print(f"\nConfusion Matrix:")
    print(cm)
    results_df = pd.DataFrame([metrics])
    results_df.to_csv('vulexplainer_results.csv', index=False)
    print(f"\nResults saved to vulexplainer_results.csv")
    per_class_results = []
    for class_id in range(NUM_CLASSES):
        class_mask = np.array(labels) == class_id
        if class_mask.sum() > 0:
            class_preds = np.array(predictions)[class_mask]
            class_labels = np.array(labels)[class_mask]
            class_acc = accuracy_score(class_labels, class_preds)
            class_prec, class_rec, class_f1, _ = precision_recall_fscore_support(
                class_labels, class_preds, average='binary', pos_label=class_id, zero_division=0
            )
            per_class_results.append({
                'Class': class_id,
                'Accuracy': class_acc,
                'Precision': class_prec,
                'Recall': class_rec,
                'F1': class_f1,
                'Samples': class_mask.sum()
            })
    per_class_df = pd.DataFrame(per_class_results)
    per_class_df.to_csv('vulexplainer_per_class_results.csv', index=False)
    print(f"Per-class results saved to vulexplainer_per_class_results.csv")
    per_group_results = []
    for group_id in group_to_labels:
        group_labels_list = group_to_labels[group_id]
        group_mask = [l in group_labels_list for l in labels]
        group_labels_filtered = [l for l, m in zip(labels, group_mask) if m]
        group_preds_filtered = [p for p, m in zip(predictions, group_mask) if m]
        if len(group_labels_filtered) > 0:
            acc = accuracy_score(group_labels_filtered, group_preds_filtered)
            prec, rec, f1_g, _ = precision_recall_fscore_support(
                group_labels_filtered, group_preds_filtered, average='weighted', zero_division=0
            )
            per_group_results.append({
                'Group': group_id,
                'Labels': str(group_labels_list),
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1': f1_g,
                'Samples': len(group_labels_filtered)
            })
    per_group_df = pd.DataFrame(per_group_results)
    per_group_df.to_csv('vulexplainer_per_group_results.csv', index=False)
    print(f"Per-group results saved to vulexplainer_per_group_results.csv")
    print("\n" + "="*70)
    print("VulExplainer Training Complete")
    print("="*70)
    print(f"\nFinal Summary:")
    print(f"  Overall Accuracy: {metrics['Accuracy']*100:.2f}%")
    print(f"  Overall F1 Score: {metrics['F1']*100:.2f}%")
    print(f"  Overall AUC: {metrics['AUC']*100:.2f}%")
    print(f"  MCC: {metrics['MCC']:.4f}")
    print(f"  Kappa: {metrics['Kappa']:.4f}")
    print(f"\nAll results and models saved successfully!")
    return metrics

if __name__ == "__main__":
    final_metrics = main()
